# 14 — Verify the Label Matrix

Step 1 verified the pixels against the shards. Step 2 measured the side
classifier. This is the third input to every fold in the 0.906 ensemble, and
the only one whose artifact has never been checked **in the place the training
runs read it from**.

Two different questions, and this notebook keeps them apart because conflating
them is how a pipeline gets called "verified" when it is only self-consistent:

1. **Is the shipped file the one this code produces?** Answerable, and
   answered here at every cell of the matrix.
2. **Are the labels correct?** Answerable only on the 58 radiologist-labelled
   studies, against 4,349 mined ones. Part D reports how little that can
   resolve rather than quoting 0.891 and stopping; Part F reports the shape of
   the error, which turns out to be almost entirely misses rather than false
   alarms.

**Attach, in the sidebar:**

| input | what for |
|---|---|
| competition `rsna-knee-abnormality-detection` | `train.csv` — the reports and the gold 58 |
| dataset `pssv27/knee-labels` | `train_labels.parquet`, the shipped artifact |
| dataset `pssv27/knee-labels` *(if it carries them)* | `llm_labels*.parquet`, `llm_probs*.parquet` — the extractor dumps |

**Accelerator: None.** Nothing here decodes a pixel or loads a model, so it
costs nothing against the 25 GPU-hour weekly quota. Internet is not needed.

Parts B, D, E and F need the extractor dumps. If `knee-labels` carries only the
finished matrix, those parts skip with a notice and Parts A and C still run —
and those two alone are enough to catch a wrong or stale upload, which is the
failure that would actually cost a submission.

In [ ]:
# Materialise the repo modules. Generated from source by
# tools/make_notebook.py -- do not edit here, edit src/ and regenerate.
import importlib, pathlib, sys

FILES = {
    'src/__init__.py': '',
    'src/config.py': '"""Central configuration: label names, cache geometry, and environment paths.\n\nEverything here is imported by both the training pipeline and the submission\nnotebook, so it must stay dependency-free apart from the standard library.\n"""\n\nimport os\nfrom pathlib import Path\n\n# ---------------------------------------------------------------------------\n# Labels\n# ---------------------------------------------------------------------------\n# Exact submission spelling. The apostrophe in "Baker\'s" and the spaces are\n# part of the column names -- do not normalise them anywhere that touches a\n# submission file.\nLABELS = [\n    "ACL",\n    "MCL",\n    "Medial Meniscus",\n    "Lateral Meniscus",\n    "Medial OA",\n    "Lateral OA",\n    "PF OA",\n    "Effusion",\n    "Synovitis",\n    "Baker\'s",\n    "Contusion",\n    "Fracture",\n]\n\nID_COL = "StudyInstanceUID"\nSUBMISSION_COLUMNS = [ID_COL] + LABELS\n\n# Compartment-specific labels. A horizontal flip swaps medial and lateral, so\n# any augmentation that mirrors the image must also swap these pairs.\nLATERAL_LABEL_PAIRS = [\n    ("Medial Meniscus", "Lateral Meniscus"),\n    ("Medial OA", "Lateral OA"),\n]\n\n# ---------------------------------------------------------------------------\n# Cache geometry\n# ---------------------------------------------------------------------------\n# Kept as constants so a higher-resolution variant is a config change rather\n# than a rewrite. Sizing: 224 * 224 * 24 bytes ~= 1.2 MB/series,\n# x 24,371 series ~= 30 GB, which fits Kaggle\'s 200 GB private dataset limit.\nCACHE_SIZE = 224          # in-plane resolution after resampling\nCACHE_SLICES = 24         # slices sampled per series, centred on the volume\nCACHE_DTYPE = "uint8"\n\n# Robust intensity normalisation percentiles. MRI has no standardised units,\n# so every volume is scaled against its own distribution.\nCLIP_PCT_LOW = 1.0\nCLIP_PCT_HIGH = 99.0\n\n# Canonical laterality: left knees are mirrored to match right knees so the\n# medial compartment always falls on the same side of the image.\nCANONICAL_SIDE = "R"\n\n# Mirroring is applied at load time, not baked into the cache.\n#\n# The side tag is blank on most series in this dataset and the series\n# descriptions are anonymised, so roughly 42% of series have no recoverable\n# side from metadata alone. Freezing a mirror decision into 30 GB of cache\n# would mean rebuilding all of it every time side detection improves -- and it\n# is expected to improve, via study-level propagation and then a classifier\n# trained on the series whose side is known. The cache therefore stores volumes\n# in their acquired orientation with the side recorded alongside, and the data\n# loader flips on read, which costs nothing.\nMIRROR_IN_CACHE = False\n\n# ---------------------------------------------------------------------------\n# Paths\n# ---------------------------------------------------------------------------\nCOMPETITION = "rsna-knee-abnormality-detection"\nPROJECT_ROOT = Path(__file__).resolve().parent.parent\n\n\n# What a competition data root must contain to be recognised.\n_MARKERS = ("train_series", "test_series")\n\n\ndef _detect_data_root() -> Path:\n    """Locate the competition data across Kaggle, Colab, and local runs.\n\n    An explicit KNEE_DATA_ROOT wins. Otherwise Kaggle\'s input directory is\n    searched by content rather than by name: the mount is not reliably at\n    /kaggle/input/<competition-slug>, and guessing the name silently falls\n    through to the local fallback, which then fails much later with a confusing\n    path in the message.\n    """\n    env = os.environ.get("KNEE_DATA_ROOT")\n    if env:\n        return Path(env)\n\n    kaggle_input = Path("/kaggle/input")\n    if kaggle_input.exists():\n        exact = kaggle_input / COMPETITION\n        if exact.exists():\n            return exact\n        # Search two levels: Kaggle mounts competition data under\n        # /kaggle/input/competitions/<slug>/ as well as directly under\n        # /kaggle/input/<slug>/, and which one you get is not predictable.\n        try:\n            for child in sorted(kaggle_input.iterdir()):\n                if not child.is_dir():\n                    continue\n                if any((child / m).exists() for m in _MARKERS):\n                    return child\n                for grandchild in sorted(child.iterdir()):\n                    if grandchild.is_dir() and any(\n                            (grandchild / m).exists() for m in _MARKERS):\n                        return grandchild\n        except OSError:\n            pass\n\n    return PROJECT_ROOT / "data"\n\n\nDATA_ROOT = _detect_data_root()\nMETA_DIR = PROJECT_ROOT / "data" / "meta"\nCACHE_DIR = Path(os.environ.get("KNEE_CACHE_DIR", PROJECT_ROOT / "data" / "cache"))\n\nTRAIN_CSV = META_DIR / "train.csv"\nTRAIN_SERIES_CSV = META_DIR / "train_series.csv"\nTEST_CSV = META_DIR / "test.csv"\nTEST_SERIES_CSV = META_DIR / "test_series.csv"\nSAMPLE_SUBMISSION_CSV = META_DIR / "sample_submission.csv"\n\nINDEX_PARQUET = META_DIR / "index.parquet"\n\nON_KAGGLE = Path("/kaggle").exists()\n',
    'src/data/__init__.py': '',
    'src/data/dicom_io.py': '"""DICOM reading, geometric slice ordering, and laterality resolution.\n\nTwo operations here are load-bearing and fail silently if done naively:\n\n1. **Slice ordering.** InstanceNumber is not reliable across the 19 contributing\n   institutions. Slices are ordered by projecting ImagePositionPatient onto the\n   slice normal derived from ImageOrientationPatient, which is defined by the\n   scanner geometry rather than by convention.\n\n2. **Laterality.** Four of the twelve labels are compartment-specific\n   (medial vs. lateral). In an image of a left knee the medial compartment sits\n   on the opposite side of the frame from a right knee, so left knees must be\n   mirrored into a canonical orientation before a model can learn the\n   distinction.\n"""\n\nimport re\nfrom pathlib import Path\nfrom typing import Iterable, List, Optional, Sequence\n\nimport numpy as np\nimport pydicom\n\n# Header fields pulled during the index scan. Kept explicit so the parquet\n# schema is stable and the scan stays header-only.\nHEADER_FIELDS = [\n    "StudyInstanceUID",\n    "SeriesInstanceUID",\n    "SOPInstanceUID",\n    "Rows",\n    "Columns",\n    "PixelSpacing",\n    "ImagePositionPatient",\n    "ImageOrientationPatient",\n    "SliceThickness",\n    "SpacingBetweenSlices",\n    "EchoTime",\n    "RepetitionTime",\n    "InversionTime",\n    "ScanningSequence",\n    "SequenceVariant",\n    # (0018,0010). The one header that states contrast administration\n    # outright, and the index built before 2026-08-31 does not carry it --\n    # so the contrast audit has to fall back on typed descriptions, 17.7% of\n    # which are anonymised away. Absence of a string is not absence of\n    # contrast; absence of this tag is at least a direct question asked.\n    "ContrastBolusAgent",\n    "SeriesDescription",\n    "ProtocolName",\n    "BodyPartExamined",\n    "Manufacturer",\n    "ManufacturerModelName",\n    "MagneticFieldStrength",\n    "ImageLaterality",\n    "Laterality",\n    "InstanceNumber",\n]\n\n# DICOM DS/IS values carry their original string representation, which would\n# land in the index as object dtype. Cast these explicitly.\nNUMERIC_FIELDS = {\n    "Rows",\n    "Columns",\n    "SliceThickness",\n    "SpacingBetweenSlices",\n    "EchoTime",\n    "RepetitionTime",\n    "InversionTime",\n    "MagneticFieldStrength",\n    "InstanceNumber",\n}\n\n_LEFT_PATTERN = re.compile(\n    r"\\b(l|lt|left|links|izquierda|izq|gauche|sinistra|sin|links?e)\\b",\n    re.IGNORECASE,\n)\n_RIGHT_PATTERN = re.compile(\n    r"\\b(r|rt|right|rechts|derecha|der|droite|destra|dx)\\b",\n    re.IGNORECASE,\n)\n\n\ndef read_header(path) -> dict:\n    """Read one DICOM\'s headers without decoding pixel data."""\n    ds = pydicom.dcmread(str(path), stop_before_pixels=True, force=True)\n    row = {"path": str(path)}\n    for field in HEADER_FIELDS:\n        value = getattr(ds, field, None)\n        if value is None:\n            row[field] = None\n        elif isinstance(value, pydicom.multival.MultiValue):\n            # Geometry fields are numeric; ScanningSequence and friends are\n            # multi-valued strings, so fall back to a joined token.\n            try:\n                row[field] = [float(v) for v in value]\n            except (TypeError, ValueError):\n                row[field] = "\\\\".join(str(v) for v in value)\n        elif field in NUMERIC_FIELDS:\n            try:\n                row[field] = float(value)\n            except (TypeError, ValueError):\n                row[field] = None\n        elif isinstance(value, (int, float, str)):\n            row[field] = str(value) if isinstance(value, str) else value\n        else:\n            row[field] = str(value)\n    return row\n\n\ndef slice_normal(orientation: Optional[Sequence[float]]) -> np.ndarray:\n    """Unit normal of the image plane, from the two direction cosines.\n\n    Falls back to the z axis when orientation is missing, which keeps a\n    malformed series ordered by patient-superior position rather than crashing.\n    """\n    if orientation is None or len(orientation) != 6:\n        return np.array([0.0, 0.0, 1.0])\n\n    row_cosine = np.asarray(orientation[:3], dtype=float)\n    col_cosine = np.asarray(orientation[3:], dtype=float)\n    normal = np.cross(row_cosine, col_cosine)\n\n    magnitude = np.linalg.norm(normal)\n    if magnitude == 0:\n        return np.array([0.0, 0.0, 1.0])\n    return normal / magnitude\n\n\ndef slice_position(header: dict) -> float:\n    """Scalar position of a slice along its own stack axis."""\n    position = header.get("ImagePositionPatient")\n    if position is None or len(position) != 3:\n        # No geometry available; fall back to InstanceNumber so ordering is at\n        # least deterministic. The index scan flags these series separately.\n        number = header.get("InstanceNumber")\n        return float(number) if number is not None else 0.0\n\n    normal = slice_normal(header.get("ImageOrientationPatient"))\n    return float(np.dot(np.asarray(position, dtype=float), normal))\n\n\ndef sort_slices(headers: Iterable[dict]) -> List[dict]:\n    """Order a series\' slices along the acquisition axis."""\n    return sorted(headers, key=slice_position)\n\n\ndef has_geometry(headers: Sequence[dict]) -> bool:\n    """True when every slice carries the position and orientation tags."""\n    for header in headers:\n        position = header.get("ImagePositionPatient")\n        orientation = header.get("ImageOrientationPatient")\n        if position is None or len(position) != 3:\n            return False\n        if orientation is None or len(orientation) != 6:\n            return False\n    return True\n\n\ndef resolve_laterality(headers: Sequence[dict]) -> Optional[str]:\n    """Determine whether a series images a left or right knee.\n\n    Prefers the explicit DICOM tags, then falls back to parsing the free-text\n    description fields, which carry side markers in several languages. Returns\n    "L", "R", or None when the side cannot be established -- callers must treat\n    None as unusable rather than guessing, since a wrong side silently swaps the\n    medial and lateral labels.\n    """\n    for header in headers:\n        for field in ("ImageLaterality", "Laterality"):\n            value = header.get(field)\n            if isinstance(value, str):\n                # The tag holds "L"/"R" but also "LEFT"/"RIGHT", and is far more\n                # often present-but-empty than absent. An empty string is not a\n                # side and must not be read as one.\n                token = value.strip().upper()\n                if token in ("L", "LEFT"):\n                    return "L"\n                if token in ("R", "RIGHT"):\n                    return "R"\n\n    for field in ("SeriesDescription", "ProtocolName", "BodyPartExamined"):\n        for header in headers:\n            text = header.get(field)\n            if not isinstance(text, str):\n                continue\n            # Split on non-letters so "MR_Knie_Rechts" yields usable tokens.\n            spaced = re.sub(r"[^A-Za-zÀ-ÿ]+", " ", text)\n            left = bool(_LEFT_PATTERN.search(spaced))\n            right = bool(_RIGHT_PATTERN.search(spaced))\n            # Ambiguous strings mentioning both sides tell us nothing.\n            if left and not right:\n                return "L"\n            if right and not left:\n                return "R"\n\n    return None\n\n\ndef load_series_volume(paths: Sequence[str]) -> np.ndarray:\n    """Decode an ordered list of DICOM paths into a (slices, rows, cols) array.\n\n    Applies the rescale slope and intercept when present. Slices whose in-plane\n    shape differs from the first are skipped rather than silently broadcast.\n    """\n    slices = []\n    reference_shape = None\n    skipped = 0\n\n    for path in paths:\n        try:\n            ds = pydicom.dcmread(str(path), force=True)\n            pixels = ds.pixel_array.astype(np.float32)\n        except Exception:  # noqa: BLE001\n            # A truncated or malformed file loses one slice, not the series.\n            # The dataset contains files whose declared pixel length exceeds\n            # what is actually stored, and dropping 23 good slices over one\n            # bad one costs a whole study its imaging.\n            skipped += 1\n            continue\n\n        slope = float(getattr(ds, "RescaleSlope", 1.0) or 1.0)\n        intercept = float(getattr(ds, "RescaleIntercept", 0.0) or 0.0)\n        if slope != 1.0 or intercept != 0.0:\n            pixels = pixels * slope + intercept\n\n        if reference_shape is None:\n            reference_shape = pixels.shape\n        elif pixels.shape != reference_shape:\n            skipped += 1\n            continue\n\n        slices.append(pixels)\n\n    if not slices:\n        raise ValueError(f"no decodable slices in series ({skipped} unreadable)")\n\n    return np.stack(slices, axis=0)\n\n\ndef series_paths(series_dir) -> List[str]:\n    """All DICOM files under one series directory."""\n    return sorted(str(p) for p in Path(series_dir).glob("*.dcm"))\n\n\ndef resolve_study_laterality(series_sides: dict) -> dict:\n    """Fill in unknown series sides from other series in the same study.\n\n    A knee MRI study images one knee, so a side established on any series\n    applies to all of them. This matters because the DICOM side tags are\n    populated inconsistently across the contributing institutions -- a study\n    can carry the tag on its coronal series and omit it on the sagittals.\n\n    Report text is deliberately not used as a source here. Reports exist only\n    for training studies, so recovering a side from them would mirror the\n    training volumes and leave the test volumes untouched -- a train/test skew\n    that is worse than the missing side it fixes.\n\n    Args:\n        series_sides: {study_uid: {series_uid: "L" | "R" | None}}\n\n    Returns:\n        The same structure with unknown entries filled where the study allows.\n        Studies whose series disagree are left untouched, since a conflict\n        means the assumption of one knee per study does not hold there.\n    """\n    resolved = {}\n    for study_uid, series in series_sides.items():\n        known = {side for side in series.values() if side in ("L", "R")}\n\n        if len(known) == 1:\n            side = known.pop()\n            resolved[study_uid] = {uid: side for uid in series}\n        else:\n            # No evidence, or contradictory evidence. Guessing would silently\n            # swap the medial and lateral labels on four of twelve findings.\n            resolved[study_uid] = dict(series)\n\n    return resolved\n\n\ndef laterality_coverage(series_sides: dict) -> dict:\n    """Counts before and after study-level propagation, for auditing."""\n    before = sum(1 for s in series_sides.values() for v in s.values()\n                 if v in ("L", "R"))\n    total = sum(len(s) for s in series_sides.values())\n\n    filled = resolve_study_laterality(series_sides)\n    after = sum(1 for s in filled.values() for v in s.values()\n                if v in ("L", "R"))\n\n    conflicts = sum(\n        1 for s in series_sides.values()\n        if len({v for v in s.values() if v in ("L", "R")}) > 1\n    )\n\n    return {\n        "total_series": total,\n        "known_before": before,\n        "known_after": after,\n        "recovered": after - before,\n        "conflicting_studies": conflicts,\n    }\n',
    'src/data/cache.py': '"""Turn a raw DICOM series into a compact, model-ready uint8 volume.\n\nThe same code path runs during cache construction and at submission time. Any\ndivergence between the two produces a train/test preprocessing skew, which is\ninvisible in local validation and only shows up as a collapsed leaderboard\nscore -- so callers must import from here rather than reimplementing.\n\nOutput per series: (CACHE_SLICES, CACHE_SIZE, CACHE_SIZE) uint8, in the\norientation it was acquired in. Mirroring to a canonical side happens at load\ntime rather than here -- see config.MIRROR_IN_CACHE for why.\n"""\n\nfrom typing import Dict, List, Optional, Sequence\n\nimport cv2\nimport numpy as np\n\nfrom src import config\nfrom src.data import dicom_io\n\n\ndef normalize_intensity(volume: np.ndarray) -> np.ndarray:\n    """Scale a volume to uint8 using its own robust percentile range.\n\n    MRI carries no standardised units -- intensity depends on scanner, field\n    strength, coil, and whether fat suppression was applied -- so each volume is\n    scaled against its own distribution rather than a global constant.\n    """\n    finite = volume[np.isfinite(volume)]\n    if finite.size == 0:\n        return np.zeros(volume.shape, dtype=np.uint8)\n\n    low, high = np.percentile(finite, [config.CLIP_PCT_LOW, config.CLIP_PCT_HIGH])\n    if not np.isfinite(low) or not np.isfinite(high) or high <= low:\n        # Flat or degenerate volume; nothing to stretch.\n        return np.zeros(volume.shape, dtype=np.uint8)\n\n    scaled = (np.clip(volume, low, high) - low) / (high - low)\n    return (scaled * 255.0).round().astype(np.uint8)\n\n\ndef resize_volume(volume: np.ndarray, size: int = None) -> np.ndarray:\n    """Resample every slice to size x size with area/linear interpolation."""\n    size = size or config.CACHE_SIZE\n    output = np.empty((volume.shape[0], size, size), dtype=volume.dtype)\n\n    for i, plane in enumerate(volume):\n        # Area interpolation avoids the aliasing that bilinear introduces when\n        # downsampling 800px acquisitions; linear is correct when upsampling.\n        interp = cv2.INTER_AREA if plane.shape[0] > size else cv2.INTER_LINEAR\n        output[i] = cv2.resize(plane, (size, size), interpolation=interp)\n\n    return output\n\n\ndef select_slices(count: int, target: int = None) -> List[int]:\n    """Indices of `target` slices spread evenly across a stack of `count`.\n\n    Short series are handled by repeating edge slices rather than zero-padding,\n    so the volume stays anatomically continuous.\n    """\n    target = target or config.CACHE_SLICES\n    if count <= 0:\n        raise ValueError("cannot select slices from an empty series")\n    if count == target:\n        return list(range(count))\n\n    positions = np.linspace(0, count - 1, target)\n    return [int(round(p)) for p in positions]\n\n\ndef mirror_axis(orientation: Optional[Sequence[float]]) -> int:\n    """Which array axis runs medial-to-lateral, from the scanner geometry.\n\n    Mirroring a knee means reflecting through the patient\'s left-right axis,\n    and which array axis that corresponds to depends on the acquisition plane:\n\n        coronal / axial   left-right lies in-plane  -> flip columns (axis 2)\n        sagittal          left-right is the slice axis -> reverse slices (axis 0)\n\n    A blanket column flip is wrong for sagittal series. It leaves medial and\n    lateral exactly where they were -- they are separated across slices, not\n    across the image -- and instead reflects anterior-posterior, moving the\n    patella to the back of the knee.\n\n    Returns 0, 1, or 2: the axis of a (slices, rows, cols) volume to reverse.\n    """\n    left_right = np.array([1.0, 0.0, 0.0])\n\n    if orientation is None or len(orientation) != 6:\n        # No geometry to reason from. Columns is the common case (coronal and\n        # axial together outnumber sagittal), but the caller should prefer not\n        # to mirror at all when geometry is missing.\n        return 2\n\n    width = np.asarray(orientation[:3], dtype=float)    # increasing column index\n    height = np.asarray(orientation[3:], dtype=float)   # increasing row index\n    normal = dicom_io.slice_normal(orientation)\n\n    alignments = [\n        (abs(float(normal @ left_right)), 0),\n        (abs(float(height @ left_right)), 1),\n        (abs(float(width @ left_right)), 2),\n    ]\n    return max(alignments)[1]\n\n\ndef canonicalize_side(volume: np.ndarray, side: Optional[str],\n                      orientation: Optional[Sequence[float]] = None) -> np.ndarray:\n    """Mirror left knees so the medial compartment is always on one side.\n\n    Four labels are compartment-specific (Medial/Lateral Meniscus, Medial/\n    Lateral OA). Without this the model would have to infer the side before it\n    could place a finding, from a cue that is not reliably in the image.\n\n    The axis reflected is chosen from the acquisition geometry -- see\n    mirror_axis -- because it differs between sagittal and coronal series.\n    """\n    if side is None:\n        # Unknown side: leave untouched and let the caller decide. Guessing\n        # would silently swap four of the twelve labels.\n        return volume\n    if side.upper() == config.CANONICAL_SIDE:\n        return volume\n\n    axis = mirror_axis(orientation)\n    return np.flip(volume, axis=axis).copy()\n\n\ndef submission_indices(count: int, slices: int) -> List[int]:\n    """Raw slice indices a `slices`-deep submission actually reads.\n\n    **Not** `select_slices(count, slices)`, and the difference is the whole\n    reason this exists. The submission decodes `CACHE_SLICES` and then\n    subsamples with `np.linspace(0, CACHE_SLICES - 1, TUNED_SLICES)`; picking\n    `slices` evenly across the raw stack in one step lands on *different\n    files* for 41 of the 44 series lengths between 16 and 60. The model would\n    then be scored on anatomy it was not trained to read, and nothing would\n    raise -- the shapes agree either way. That is the same failure the\n    `volumes[:MAX_SERIES]` head truncation already cost this project.\n\n    So the two selections are composed rather than collapsed: the coarse pick\n    the cache would have made, then the fine pick the model applies to it.\n    """\n    coarse = select_slices(count)\n    if slices == config.CACHE_SLICES:\n        return coarse\n    return [coarse[i] for i in select_slices(config.CACHE_SLICES, slices)]\n\n\ndef build_series_volume(headers: Sequence[dict],\n                        slices: Optional[int] = None,\n                        size: Optional[int] = None) -> Dict:\n    """Full pipeline for one series: sort, decode, normalise, resize.\n\n    Mirroring is applied only when config.MIRROR_IN_CACHE is set, which it is\n    not by default. The recorded `laterality` is what the loader mirrors on.\n    Returns the volume plus the provenance needed to audit it later.\n\n    `slices` exists for the submission, which reads only `TUNED_SLICES` of the\n    `CACHE_SLICES` this decodes: at 16 of 24 that is a third of the pixel\n    decoding done for images the model never sees, and inference time is a\n    *scored quantity* in the Efficiency Prize. It defaults to None, so cache\n    construction is unchanged and every existing shard stays valid.\n\n    **It is not bit-identical to decoding the full depth.**\n    `normalize_intensity` scales each volume by its own 1st/99th percentiles,\n    computed over whatever was decoded, so a shallower stack shifts the\n    intensity mapping slightly. The slices themselves are the same files --\n    see `submission_indices` -- but the uint8 values differ a little, and the\n    submission notebook measures that difference rather than assuming it away.\n    """\n    if not headers:\n        raise ValueError("no headers supplied")\n    if size is not None and (not isinstance(size, int) or isinstance(size, bool) or size < 1):\n        raise ValueError("size must be a positive integer")\n\n    depth = config.CACHE_SLICES if slices is None else slices\n    ordered = dicom_io.sort_slices(headers)\n    first_header = ordered[0]\n    side = dicom_io.resolve_laterality(ordered)\n    geometry_ok = dicom_io.has_geometry(ordered)\n\n    indices = submission_indices(len(ordered), depth)\n    selected = [ordered[i] for i in indices]\n\n    raw = dicom_io.load_series_volume([h["path"] for h in selected])\n\n    # Slices skipped as unreadable leave the volume short of the requested\n    # depth, and the cache is a fixed-shape array. Re-select across whatever\n    # decoded so the depth is exact regardless of how many files were dropped.\n    if raw.shape[0] != depth:\n        raw = raw[select_slices(raw.shape[0], depth)]\n\n    resized = resize_volume(raw.astype(np.float32), size=size)\n    normalized = normalize_intensity(resized)\n\n    # Left as acquired by default; the loader mirrors using the recorded side.\n    # See config.MIRROR_IN_CACHE for why this is not baked in here.\n    volume = (canonicalize_side(normalized, side,\n                                first_header.get("ImageOrientationPatient"))\n              if config.MIRROR_IN_CACHE else normalized)\n\n    first = ordered[0]\n    return {\n        "volume": volume,\n        "StudyInstanceUID": first.get("StudyInstanceUID"),\n        "SeriesInstanceUID": first.get("SeriesInstanceUID"),\n        "n_slices_raw": len(ordered),\n        "laterality": side,\n        "mirrored": bool(config.MIRROR_IN_CACHE and side is not None\n                         and side.upper() != config.CANONICAL_SIDE),\n        "geometry_ok": geometry_ok,\n        "rows": int(first.get("Rows") or 0),\n        "cols": int(first.get("Columns") or 0),\n    }\n\n\ndef build_series_record(item):\n    """Picklable worker for a process pool: (series_uid, headers) -> result.\n\n    Defined at module level deliberately. A worker defined inside a notebook\n    cell cannot be pickled by name, so ProcessPoolExecutor fails under the\n    spawn start method and survives under fork only by accident. Keeping it\n    importable makes the cache build work the same way everywhere.\n\n    Failures are returned rather than raised: one unreadable series must not\n    end a ninety-minute run, and the manifest records what to investigate.\n    """\n    series_uid, headers = item\n    try:\n        result = build_series_volume(headers)\n        return series_uid, result["volume"], {\n            "SeriesInstanceUID": series_uid,\n            "StudyInstanceUID": result["StudyInstanceUID"],\n            "n_slices_raw": result["n_slices_raw"],\n            "laterality": result["laterality"],\n            "mirrored": result["mirrored"],\n            "geometry_ok": result["geometry_ok"],\n            "rows": result["rows"],\n            "cols": result["cols"],\n            "error": None,\n        }\n    except Exception as exc:  # noqa: BLE001 - recorded, not raised\n        study_uid = headers[0].get("StudyInstanceUID") if headers else None\n        return series_uid, None, {\n            "SeriesInstanceUID": series_uid,\n            "StudyInstanceUID": study_uid,\n            "n_slices_raw": len(headers) if headers else 0,\n            "laterality": None,\n            "mirrored": False,\n            "geometry_ok": False,\n            "rows": 0,\n            "cols": 0,\n            "error": str(exc)[:200],\n        }\n',
    'src/data/index.py': '"""Header-only scan of the DICOM tree into a single parquet index.\n\nRuns on Kaggle, where the competition data is mounted. Reads no pixel data, so\nthe cost is dominated by file-system traversal rather than decoding. The\nresulting index drives cache construction and lets series be filtered and\naudited without touching the 570 GB of imaging again.\n\nUsage (in a Kaggle notebook):\n\n    from src.data.index import build_index\n    build_index("train_series", out_path="/kaggle/working/index_train.parquet")\n"""\n\nimport argparse\nimport os\nfrom concurrent.futures import ThreadPoolExecutor\nfrom pathlib import Path\nfrom typing import Iterator, List, Optional\n\nimport pandas as pd\n\nfrom src import config\nfrom src.data import dicom_io\n\n\ndef iter_dicom_paths(root) -> Iterator[Path]:\n    """Yield every .dcm under root/study/series/ without materialising a list."""\n    root = Path(root)\n    for study_dir in sorted(root.iterdir()):\n        if not study_dir.is_dir():\n            continue\n        for series_dir in sorted(study_dir.iterdir()):\n            if not series_dir.is_dir():\n                continue\n            for path in sorted(series_dir.glob("*.dcm")):\n                yield path\n\n\ndef scan_paths(paths: List[Path], workers: int = 8) -> pd.DataFrame:\n    """Read headers for a batch of files in parallel.\n\n    Header reads are I/O bound, so threads are the right tool -- the GIL is\n    released during file reads and a process pool would only add overhead.\n    """\n    def safe_read(path):\n        try:\n            return dicom_io.read_header(path)\n        except Exception as exc:  # noqa: BLE001 - one bad file must not stop a scan\n            return {"path": str(path), "error": str(exc)}\n\n    with ThreadPoolExecutor(max_workers=workers) as pool:\n        rows = list(pool.map(safe_read, paths))\n\n    return pd.DataFrame(rows)\n\n\ndef build_index(\n    split: str = "train_series",\n    out_path: Optional[str] = None,\n    data_root=None,\n    workers: int = 8,\n    batch_size: int = 5000,\n    limit: Optional[int] = None,\n) -> pd.DataFrame:\n    """Scan one split\'s DICOM tree and write a parquet index.\n\n    Processes in batches so a long scan reports progress and so peak memory\n    stays flat regardless of how many files the split holds.\n    """\n    root = Path(data_root or config.DATA_ROOT) / split\n    if not root.exists():\n        raise FileNotFoundError(f"{root} does not exist")\n\n    try:\n        from tqdm.auto import tqdm\n    except ImportError:  # progress is a convenience, never a requirement\n        tqdm = None\n\n    frames = []\n    batch: List[Path] = []\n    scanned = 0\n\n    # The tree is walked lazily, so the total is unknown until the end. tqdm\n    # counts files without a total rather than forcing a full listing first,\n    # which on this dataset would mean stat-ing ~820,000 paths up front.\n    progress = tqdm(total=limit, unit="file", desc=f"scanning {split}") if tqdm else None\n\n    for path in iter_dicom_paths(root):\n        batch.append(path)\n        if limit is not None and len(batch) + scanned >= limit:\n            break\n        if len(batch) >= batch_size:\n            frames.append(scan_paths(batch, workers=workers))\n            scanned += len(batch)\n            if progress is not None:\n                progress.update(len(batch))\n                progress.set_postfix(series=int(frames[-1]["SeriesInstanceUID"].nunique()))\n            else:\n                print(f"  scanned {scanned} files", flush=True)\n            batch = []\n\n    if batch:\n        frames.append(scan_paths(batch, workers=workers))\n        scanned += len(batch)\n        if progress is not None:\n            progress.update(len(batch))\n\n    if progress is not None:\n        progress.close()\n\n    if not frames:\n        raise RuntimeError(f"no DICOM files found under {root}")\n\n    index = pd.concat(frames, ignore_index=True)\n    print(f"scanned {scanned} files across {index[\'SeriesInstanceUID\'].nunique()} series")\n\n    if "error" in index.columns:\n        failures = int(index["error"].notna().sum())\n        if failures:\n            print(f"WARNING: {failures} files failed to parse")\n\n    if out_path:\n        os.makedirs(Path(out_path).parent, exist_ok=True)\n        index.to_parquet(out_path, index=False)\n        print(f"wrote {out_path}")\n\n    return index\n\n\ndef summarize(index: pd.DataFrame) -> pd.DataFrame:\n    """Per-series roll-up: slice count, geometry completeness, laterality.\n\n    Surfaces the series that need attention -- missing geometry, unresolvable\n    side -- before an hour of cache construction is spent on them.\n    """\n    records = []\n    for series_uid, group in index.groupby("SeriesInstanceUID", sort=False):\n        headers = group.to_dict("records")\n        records.append({\n            "StudyInstanceUID": headers[0].get("StudyInstanceUID"),\n            "SeriesInstanceUID": series_uid,\n            "n_slices": len(headers),\n            "geometry_ok": dicom_io.has_geometry(headers),\n            "laterality": dicom_io.resolve_laterality(headers),\n            "rows": headers[0].get("Rows"),\n            "cols": headers[0].get("Columns"),\n        })\n\n    return pd.DataFrame(records)\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--split", default="train_series")\n    parser.add_argument("--out", default=None)\n    parser.add_argument("--workers", type=int, default=8)\n    parser.add_argument("--limit", type=int, default=None)\n    args = parser.parse_args()\n\n    build_index(split=args.split, out_path=args.out,\n                workers=args.workers, limit=args.limit)\n\n\nif __name__ == "__main__":\n    main()\n',
    'src/labels/__init__.py': '',
    'src/labels/lexicon.py': '"""Rule-based extraction of the twelve findings from multilingual reports.\n\nThe corpus spans roughly eleven languages -- English (39%), Greek, Croatian/\nSerbian, Spanish, Turkish, German, Bulgarian/Russian, Dutch, French, Italian,\nPortuguese. Rather than a language x finding cross-product, this module keys on\nthe Greek and Latin stems that clinical vocabulary shares across all of them\n(menisc- / menisk- / μηνίσκ- / мениск-), with language-specific additions only\nwhere the local term is genuinely unrelated (Turkish "kıkırdak" for cartilage).\n\nEach finding resolves to one of three states:\n\n    1     an unnegated mention above the severity floor\n    0     mentioned but negated, or present only below the severity floor\n    None  never mentioned -- which is *not* the same as absent, because\n          reports narrate positives and omit much else\n\nCallers must preserve the None state. Collapsing it to 0 injects systematic\nnoise into exactly the findings that are least often narrated.\n"""\n\nimport re\nimport unicodedata\nfrom typing import Dict, List, Optional\n\nfrom src import config\n\n# ---------------------------------------------------------------------------\n# Cues\n# ---------------------------------------------------------------------------\n# Negation, across every language present in the corpus. Matched as whole words\n# within a bounded window before the finding term.\nNEGATION_CUES = [\n    # English / Latin-script general\n    "no", "not", "without", "absent", "negative", "unremarkable", "intact",\n    "normal", "preserved", "free of", "rule out", "ruled out",\n    # Spanish / Portuguese\n    "sin", "no", "ausencia", "ausente", "conservado", "conservada", "integro",\n    "íntegro", "integra", "íntegra", "normales", "sem", "nao", "não",\n    # Dutch\n    "geen", "niet", "zonder", "intacte", "ongestoord",\n    # German\n    "kein", "keine", "keinen", "nicht", "ohne", "unauffällig", "regelrecht",\n    "unauffaellig",\n    # French\n    "pas", "sans", "absence", "aucun", "aucune", "normale",\n    # Italian\n    "non", "senza", "assenza", "nella norma",\n    # Turkish\n    "yok", "yoktur", "izlenmemiştir", "izlenmemistir", "saptanmamıştır",\n    "değil", "degil", "normaldir", "korunmuş", "korunmus",\n    # Croatian / Serbian / Bosnian\n    "bez", "nema", "nije", "uredan", "uredna", "uredno", "očuvan", "ocuvan",\n    # Bulgarian / Russian\n    "без", "нет", "няма", "не", "отсутств", "интактн", "сохранен",\n    # Greek\n    "χωρίς", "δεν", "ουδεμία", "ουδέν", "φυσιολογικ", "ακέραι",\n]\n\n# Hedged or sub-threshold mentions. Ground truth was adjudicated against\n# explicit severity thresholds, so "trace effusion" and "large effusion" are\n# not the same label.\nMINIMAL_CUES = [\n    "minimal", "mínimo", "minimo", "minima", "mínima", "trace", "tiny",\n    "slight", "small", "mild", "leve", "ligero", "ligera", "discreto",\n    "discreta", "gering", "geringe", "klein", "kleine", "wenig", "licht",\n    "lichte", "petit", "petite", "faible", "lieve", "piccolo", "az", "küçük",\n    "kucuk", "hafif", "minimalan", "mali", "blag", "малък", "минимален",\n    "незначител", "ελάχιστ", "μικρ", "ήπι",\n]\n\nUNCERTAIN_CUES = [\n    "possible", "possibly", "suspicious", "suspected", "probable", "probably",\n    "questionable", "equivocal", "cannot exclude", "cannot be excluded",\n    "may represent", "posible", "sospecha", "sospechoso", "dudoso",\n    "verdacht", "möglich", "moglich", "fraglich", "mogelijk", "verdenking",\n    "suspect", "possible", "suspicion", "şüpheli", "supheli", "olası", "olasi",\n    "sumnja", "moguć", "mogu", "възможн", "подозр", "πιθαν", "ύποπτ",\n]\n\n# Negation that *follows* the finding term. Turkish is verb-final, so\n# "eklem ici sivi yoktur" negates with a trailing word and a backwards-only\n# search misses it entirely -- that is a tenth of this corpus. English and\n# several others do the same with trailing adjectives ("meniscus intact").\n# Kept deliberately narrower than NEGATION_CUES: a forward search over the full\n# cue list would let an unrelated "normal" in the same clause cancel a real\n# finding.\nPOSTFIX_NEGATION_CUES = [\n    # Turkish\n    "yok", "yoktur", "izlenmemiştir", "izlenmemistir", "saptanmamıştır",\n    "saptanmamistir", "değil", "degil", "normaldir", "normaldi", "korunmuş",\n    "korunmus", "izlenmedi",\n    # English / Latin-script trailing adjectives\n    "intact", "unremarkable", "preserved", "normal", "normale", "normales",\n    "within normal limits", "is normal", "are normal",\n    # Dutch / German\n    "intacte", "unauffällig", "unauffaellig", "regelrecht", "ongestoord",\n    # Croatian / Serbian\n    "uredan", "uredna", "uredno", "očuvan", "ocuvan", "očuvana", "ocuvana",\n    # Bulgarian / Russian\n    "сохранен", "интактн", "не се визуализира", "непроменена",\n    # Greek\n    "φυσιολογικ", "ακέραι",\n]\n\n# Terms that signal a torn or damaged structure, used by the ligament and\n# meniscus findings where the anatomy word alone means nothing.\nINJURY_STEMS = [\n    "tear", "torn", "rupture", "ruptur", "disrupt", "discontinu", "avuls",\n    "sprain", "lesion", "lesión", "lesao", "lesão", "injur", "insufficien",\n    "rotura", "ruptura", "roto", "rota", "desgarro", "riss", "einriss",\n    "scheur", "letsel", "déchirure", "dechirure", "rupture", "lacerazion",\n    "yırtık", "yirtik", "rüptür", "ruptur", "kopma", "puknuć", "puknuc",\n    "ruptura", "razdor", "разрыв", "руптура", "скъсване", "увреж",\n    "ρήξη", "ρηξη", "ρήγμα",\n]\n\n\ndef _norm(text: str) -> str:\n    """Lowercase and strip combining marks, keeping non-Latin scripts intact."""\n    lowered = text.lower()\n    decomposed = unicodedata.normalize("NFD", lowered)\n    stripped = "".join(c for c in decomposed if not unicodedata.combining(c))\n    return unicodedata.normalize("NFC", stripped)\n\n\ndef _stem_pattern(stems: List[str]) -> re.Pattern:\n    """Compile stems into one alternation matching at a word boundary."""\n    escaped = sorted((re.escape(_norm(s)) for s in stems), key=len, reverse=True)\n    return re.compile(r"(?<![^\\W\\d_])(?:" + "|".join(escaped) + r")", re.UNICODE)\n\n\n# Degenerative disease stems. The three OA findings differ only by which\n# compartment the disease is localised to, so they share this anchor and are\n# separated by the localiser lists below. Anchoring on the compartment word\n# instead does not work: "medial" appears in almost every report describing\n# normal anatomy.\nOA_STEMS = [\n    "osteoarthr", "osteoarthrit", "arthrosis", "arthrose", "artrosis",\n    "artroz", "artroza", "gonarthros", "gonartros", "gonartroz",\n    "degenerative change", "degenerative disease", "cambios degenerativos",\n    "degenerativne", "degenerativ", "chondropath", "condropat", "chondromalac",\n    "condromalac", "chondral loss", "chondral thinning", "cartilage loss",\n    "cartilage thinning", "perdida de cartilago", "adelgazamiento del cartilago",\n    "knorpelverlust", "knorpelschaden", "kraakbeenverlies", "kikirdak kayb",\n    "kikirdak incelme", "hrskavice", "denudacij", "osteofit", "osteophyt",\n    "espolon", "spur formation", "остеоартр", "артроз", "остеофит",\n    "хондромалац", "изтънен", "οστεοαρθρ", "αρθριτ", "χονδροπαθ",\n    "χονδρομαλακ", "οστεοφυτ",\n]\n\nMEDIAL_LOCALIZERS = [\n    "medial", "media", "mediale", "medialen", "medialer", "medialne",\n    "medijaln", "interno", "interna", "innen", "innenseit", "binnen",\n    "ic kompartman", "медиальн", "медиалн", "вътреш", "εσω", "εσωτερικ",\n]\n\nLATERAL_LOCALIZERS = [\n    "lateral", "laterale", "lateralen", "lateraler", "lateraln", "externo",\n    "externa", "aussen", "buiten", "dis kompartman", "латеральн", "латералн",\n    "външ", "εξω", "εξωτερικ",\n]\n\nPF_LOCALIZERS = [\n    "patellofemoral", "patelofemoral", "femoropatellar", "femoropatelar",\n    "retropatellar", "retropatelar", "patellar", "patelar", "patella",\n    "rotulian", "rotuliana", "rotuliano", "trochlea", "troclea",\n    "femoropatelarn", "patellofemoraln", "пателлофеморальн", "пателофеморалн",\n    "надколенник", "капачка", "επιγονατιδ", "επιγονατιδομηριαι",\n]\n\n# ---------------------------------------------------------------------------\n# Finding definitions\n# ---------------------------------------------------------------------------\n# `anatomy` locates the structure; `requires_injury` marks findings where the\n# structure being named is not itself abnormal (a meniscus is always present --\n# only a torn one is a positive label).\nFINDINGS = {\n    "ACL": {\n        "anatomy": ["acl", "lca", "anterior cruciate", "ligamento cruzado anterior",\n                    "voorste kruisband", "vorderes kreuzband", "kreuzband",\n                    "ligament croise anterieur", "legamento crociato anteriore",\n                    "on capraz bag", "onc apraz", "capraz bag", "prednji ukrizeni",\n                    "prednja ukrstena", "predna krstna", "передня крестообразн",\n                    "предна кръстна", "крестообразн", "prosthios chiastos",\n                    "προσθιο χιαστο", "χιαστο"],\n        "requires_injury": True,\n    },\n    "MCL": {\n        "anatomy": ["mcl", "medial collateral", "ligamento colateral medial",\n                    "ligamento lateral interno", "mediale collaterale",\n                    "innenband", "mediales kollateralband", "mediale band",\n                    "ligament collateral medial", "ic kollateral",\n                    "medial kollateral", "medijalni kolateralni",\n                    "медиальн коллатеральн", "вътрешна колатерал",\n                    "εσω πλαγιο", "εσωτερικο πλαγιο"],\n        "requires_injury": True,\n    },\n    "Medial Meniscus": {\n        "anatomy": ["medial menisc", "menisco medial", "menisco interno",\n                    "mediale meniscus", "innenmeniskus", "medialer meniskus",\n                    "menisque interne", "menisque medial", "menisco mediale",\n                    "medial menisk", "ic menisk", "medijalni meniskus",\n                    "medijalnog meniskusa", "медиальн мениск",\n                    "вътрешния менискус", "εσω μηνισκ"],\n        "requires_injury": True,\n    },\n    "Lateral Meniscus": {\n        "anatomy": ["lateral menisc", "menisco lateral", "menisco externo",\n                    "laterale meniscus", "aussenmeniskus", "lateraler meniskus",\n                    "menisque externe", "menisque lateral", "menisco laterale",\n                    "lateral menisk", "dis menisk", "lateralni meniskus",\n                    "латеральн мениск", "външния менискус", "εξω μηνισκ"],\n        "requires_injury": True,\n    },\n    # The OA family anchors on the disease stem and requires a compartment\n    # localiser nearby, within a tighter window than other findings use.\n    "Medial OA": {\n        "anatomy": OA_STEMS,\n        "requires_injury": False,\n        "context": MEDIAL_LOCALIZERS,\n        "context_window": 55,\n        "context_is_localizer": True,\n    },\n    "Lateral OA": {\n        "anatomy": OA_STEMS,\n        "requires_injury": False,\n        "context": LATERAL_LOCALIZERS,\n        "context_window": 55,\n        "context_is_localizer": True,\n    },\n    "PF OA": {\n        "anatomy": OA_STEMS,\n        "requires_injury": False,\n        "context": PF_LOCALIZERS,\n        "context_window": 55,\n        "context_is_localizer": True,\n    },\n    "Effusion": {\n        "anatomy": ["effusion", "efusion", "derrame", "hydrops", "hidrartros",\n                    "gelenkerguss", "erguss", "gewrichtsvocht", "vocht",\n                    "epanchement", "versamento", "efuzyon", "eklem ici sivi",\n                    "eklem sivisi", "izliv", "zglobni izliv", "wypot",\n                    "выпот", "излив", "ставен излив", "συλλογη υγρου",\n                    "αρθριτικη συλλογη", "ενδαρθρικη συλλογη"],\n        "requires_injury": False,\n    },\n    "Synovitis": {\n        "anatomy": ["synovit", "sinovit", "synovial thickening",\n                    "engrosamiento sinovial", "synoviale verdickung",\n                    "synoviale proliferation", "synovitis", "sinovitis",\n                    "sinovijalis", "sinovijalna", "синовит", "υμενιτιδα",\n                    "αρθρικου υμενα"],\n        "requires_injury": False,\n    },\n    "Baker\'s": {\n        "anatomy": ["baker", "popliteal cyst", "quiste poplite", "quistes poplite",\n                    "bakerse cyste", "poplitealzyste", "bakerzyste",\n                    "kyste poplite", "cisti poplitea", "poplitealna cista",\n                    "bakerova cista", "poplitealni", "baker kisti",\n                    "киста бейкера", "подколенна киста", "κυστη baker",\n                    "κυστη μπεικερ", "ιγνυακη κυστη"],\n        "requires_injury": False,\n    },\n    "Contusion": {\n        "anatomy": ["contusion", "contusion osea", "bone contusion",\n                    "bone marrow edema", "bone marrow oedema", "marrow edema",\n                    "edema oseo", "edema de medula osea", "edema medular",\n                    "knochenmarksodem", "knochenodem", "knochenmarködem",\n                    "beenmergoedeem", "botoedeem", "oedeme osseux",\n                    "edema midollare", "kemik ilig odem", "kemik odem",\n                    "kontuzyon", "kostana kontuzija", "edem kosti",\n                    "костномозгов отек", "костномозъчен едем", "оστικο οιδημα",\n                    "οιδημα μυελου", "μυελικο οιδημα"],\n        "requires_injury": False,\n    },\n    "Fracture": {\n        "anatomy": ["fracture", "fractura", "fraktur", "fractuur", "frattura",\n                    "kirik", "kırık", "prelom", "fraktura", "perelom",\n                    "перелом", "фрактура", "счупване", "καταγμα", "καταγματ"],\n        "requires_injury": False,\n    },\n}\n\n# Window, in characters, searched backwards from a term for a negation cue.\nNEGATION_WINDOW = 60\n# Window searched around a term for severity and certainty cues.\nSEVERITY_WINDOW = 45\n# Window searched forward from a term for a trailing negator.\nPOSTFIX_NEGATION_WINDOW = 40\n# Window searched around an anatomy match for the injury or context stem.\nCONTEXT_WINDOW = 90\n\n_NEGATION_RE = _stem_pattern(NEGATION_CUES)\n_POSTFIX_NEGATION_RE = _stem_pattern(POSTFIX_NEGATION_CUES)\n_MINIMAL_RE = _stem_pattern(MINIMAL_CUES)\n_UNCERTAIN_RE = _stem_pattern(UNCERTAIN_CUES)\n_INJURY_RE = _stem_pattern(INJURY_STEMS)\n\n_COMPILED = {}\nfor _name, _spec in FINDINGS.items():\n    _COMPILED[_name] = {\n        "anatomy": _stem_pattern(_spec["anatomy"]),\n        "requires_injury": _spec["requires_injury"],\n        "context": _stem_pattern(_spec["context"]) if _spec.get("context") else None,\n        "context_window": _spec.get("context_window", CONTEXT_WINDOW),\n        "context_is_localizer": _spec.get("context_is_localizer", False),\n    }\n\n# Clause boundaries. Negation does not carry across them, so "no tear. Effusion\n# present" must not read as a negated effusion.\n_CLAUSE_SPLIT = re.compile(r"[.;:\\n]|\\bbut\\b|\\bhowever\\b|\\bpero\\b|\\bmaar\\b|\\baber\\b")\n\n\ndef _clause_bounds(text: str, position: int) -> tuple:\n    """Start and end of the clause containing `position`."""\n    start = 0\n    end = len(text)\n    for match in _CLAUSE_SPLIT.finditer(text):\n        if match.end() <= position:\n            start = match.end()\n        elif match.start() > position:\n            end = match.start()\n            break\n    return start, end\n\n\ndef _is_negated(text: str, position: int, length: int = 0) -> bool:\n    """True when a negation cue applies to the term at `position`.\n\n    Searches backwards over the full cue list, and forwards over the narrower\n    postfix list -- verb-final languages put the negator after the finding.\n    Both searches stop at the clause boundary so a negation cannot leak into a\n    neighbouring statement.\n    """\n    clause_start, clause_end = _clause_bounds(text, position)\n\n    window_start = max(clause_start, position - NEGATION_WINDOW)\n    if _NEGATION_RE.search(text, window_start, position):\n        return True\n\n    term_end = position + length\n    window_end = min(clause_end, term_end + POSTFIX_NEGATION_WINDOW)\n    return bool(_POSTFIX_NEGATION_RE.search(text, term_end, window_end))\n\n\ndef _severity(text: str, position: int, length: int) -> float:\n    """Confidence that the mention clears the adjudication threshold.\n\n    1.0 for an unqualified positive, lower when the report hedges the finding\n    or calls it minimal. These scores feed the fusion step as soft targets\n    rather than being thresholded here.\n    """\n    start = max(0, position - SEVERITY_WINDOW)\n    end = min(len(text), position + length + SEVERITY_WINDOW)\n    window = text[start:end]\n\n    score = 1.0\n    if _MINIMAL_RE.search(window):\n        score *= 0.35\n    if _UNCERTAIN_RE.search(window):\n        score *= 0.5\n    return score\n\n\ndef _has_context(text: str, position: int, length: int, pattern,\n                 window: int = CONTEXT_WINDOW, clause_scoped: bool = True) -> bool:\n    """True when a required injury, disease, or localiser stem sits nearby.\n\n    Injury context is clause-scoped: a tear described in one sentence must not\n    license a finding named in the next. Compartment localisers are not, because\n    reports routinely name the disease in one clause and the compartment in the\n    following one ("Gonarthrosis. Most marked laterally."). Enforcing clause\n    scope on localisers cost Lateral OA two thirds of its recall.\n    """\n    if pattern is None:\n        return True\n\n    if clause_scoped:\n        clause_start, clause_end = _clause_bounds(text, position)\n    else:\n        clause_start, clause_end = 0, len(text)\n\n    start = max(clause_start, position - window)\n    end = min(clause_end, position + length + window)\n    return bool(pattern.search(text, start, end))\n\n\ndef extract_finding(report: str, finding: str) -> Dict:\n    """Resolve one finding from one report."""\n    spec = _COMPILED[finding]\n    text = _norm(report)\n\n    context_pattern = spec["context"]\n    context_window = spec["context_window"]\n    localizer_based = spec["context_is_localizer"]\n    if spec["requires_injury"]:\n        context_pattern = _INJURY_RE\n        context_window = CONTEXT_WINDOW\n        localizer_based = False\n\n    mentioned = False\n    best = None\n\n    for match in spec["anatomy"].finditer(text):\n        position, length = match.start(), match.end() - match.start()\n        in_context = _has_context(text, position, length, context_pattern,\n                                  context_window,\n                                  clause_scoped=not localizer_based)\n\n        if localizer_based and not in_context:\n            # Degenerative disease was described, but not in this compartment.\n            # That is silence about this finding, not evidence against it --\n            # counting it as a mention would report a confident zero for a\n            # compartment the radiologist never discussed.\n            continue\n\n        mentioned = True\n\n        if not in_context:\n            # The structure is named but nothing abnormal is asserted about it,\n            # which is a normal-anatomy sentence rather than a finding.\n            continue\n        if _is_negated(text, position, length):\n            continue\n\n        score = _severity(text, position, length)\n        clause_start, clause_end = _clause_bounds(text, position)\n        candidate = {\n            "present": 1,\n            "severity": score,\n            "span": report[clause_start:clause_end].strip()[:200],\n        }\n        # Keep the most confident positive mention.\n        if best is None or score > best["severity"]:\n            best = candidate\n\n    if best is not None:\n        return best\n    if mentioned:\n        return {"present": 0, "severity": 0.0, "span": None}\n    return {"present": None, "severity": None, "span": None}\n\n\ndef extract_report(report: str) -> Dict[str, Dict]:\n    """Resolve all twelve findings from one report."""\n    if not isinstance(report, str) or not report.strip():\n        return {name: {"present": None, "severity": None, "span": None}\n                for name in config.LABELS}\n    return {name: extract_finding(report, name) for name in config.LABELS}\n\n\ndef extract_frame(reports, ids=None):\n    """Vectorise extraction over a series of reports into a DataFrame.\n\n    Returns one row per report with `<finding>` (0/1/NaN) and\n    `<finding>_severity` columns.\n    """\n    import pandas as pd\n\n    records = []\n    for report in reports:\n        result = extract_report(report)\n        row = {}\n        for name in config.LABELS:\n            row[name] = result[name]["present"]\n            row[f"{name}_severity"] = result[name]["severity"]\n        records.append(row)\n\n    frame = pd.DataFrame(records)\n    if ids is not None:\n        frame.insert(0, config.ID_COL, list(ids))\n    return frame\n',
    'src/labels/calibrate.py': '"""Score a report extractor against the 58 radiologist-labelled studies.\n\nThose 58 labels are the only trustworthy signal in the dataset, and this is\ntheir highest-value use: measuring how much a mined label can be believed,\nper finding. They are never used for training.\n\nThe per-finding AUC produced here drives the kill criterion -- a finding whose\nmined labels score below KILL_THRESHOLD is noise, and the model should predict\nthe training prior for it rather than learn from corrupted targets.\n"""\n\nfrom typing import Callable, Dict, Optional\n\nimport numpy as np\nimport pandas as pd\n\nfrom src import config\n\n# Below this AUC against gold, a finding\'s mined labels are not worth training\n# on. Chosen to sit meaningfully above chance while tolerating the noise of a\n# 58-sample estimate.\nKILL_THRESHOLD = 0.70\n\n\ndef load_gold(train_csv=None) -> pd.DataFrame:\n    """The 58 studies carrying radiologist labels."""\n    frame = pd.read_csv(train_csv or config.TRAIN_CSV)\n    labelled = frame[frame[config.LABELS[0]].notna()].copy()\n    for name in config.LABELS:\n        labelled[name] = labelled[name].astype(int)\n    return labelled.reset_index(drop=True)\n\n\ndef load_unlabelled(train_csv=None) -> pd.DataFrame:\n    """The 4,349 studies carrying only a report."""\n    frame = pd.read_csv(train_csv or config.TRAIN_CSV)\n    return frame[frame[config.LABELS[0]].isna()].reset_index(drop=True)\n\n\ndef _auc(y_true: np.ndarray, scores: np.ndarray) -> float:\n    """ROC AUC via rank statistic, so sklearn stays an optional dependency."""\n    positives = y_true == 1\n    n_pos = int(positives.sum())\n    n_neg = int((~positives).sum())\n    if n_pos == 0 or n_neg == 0:\n        return float("nan")\n\n    order = np.argsort(scores, kind="mergesort")\n    ranks = np.empty(len(scores), dtype=float)\n    ranks[order] = np.arange(1, len(scores) + 1)\n\n    # Average ranks within ties, otherwise a constant score would not score 0.5.\n    unique, inverse, counts = np.unique(scores, return_inverse=True, return_counts=True)\n    tie_sums = np.zeros(len(unique))\n    np.add.at(tie_sums, inverse, ranks)\n    ranks = (tie_sums / counts)[inverse]\n\n    return (ranks[positives].sum() - n_pos * (n_pos + 1) / 2) / (n_pos * n_neg)\n\n\ndef score_extractor(\n    extract_fn: Callable[[str, str], Dict],\n    gold: Optional[pd.DataFrame] = None,\n) -> pd.DataFrame:\n    """Per-finding reliability of an extractor against the gold studies.\n\n    `extract_fn(report, finding)` must return a dict with `present` (1/0/None)\n    and `severity` (float or None), matching the lexicon module\'s contract.\n    """\n    gold = gold if gold is not None else load_gold()\n    reports = gold["Report"].tolist()\n    records = []\n\n    for name in config.LABELS:\n        truth = gold[name].to_numpy()\n        results = [extract_fn(report, name) for report in reports]\n\n        present = np.array([\n            r["present"] if r["present"] is not None else -1 for r in results\n        ])\n        severity = np.array([\n            r["severity"] if r["severity"] is not None else 0.0 for r in results\n        ])\n\n        predicted = (present == 1).astype(int)\n        true_pos = int(((predicted == 1) & (truth == 1)).sum())\n        false_pos = int(((predicted == 1) & (truth == 0)).sum())\n        false_neg = int(((predicted == 0) & (truth == 1)).sum())\n\n        precision = true_pos / (true_pos + false_pos) if true_pos + false_pos else np.nan\n        recall = true_pos / (true_pos + false_neg) if true_pos + false_neg else np.nan\n        auc = _auc(truth, severity)\n\n        # How often a finding goes unmentioned, and how often it is truly\n        # positive when unmentioned. The second number is what fusion should\n        # use as the soft target for silence -- treating it as a hard zero is\n        # what injects systematic noise.\n        unmentioned = present == -1\n        silence_rate = float(unmentioned.mean())\n        prior_if_silent = (\n            float(truth[unmentioned].mean()) if unmentioned.any() else np.nan\n        )\n\n        records.append({\n            "finding": name,\n            "n_positive": int(truth.sum()),\n            "mention_rate": 1.0 - silence_rate,\n            "precision": precision,\n            "recall": recall,\n            "auc": auc,\n            "prior_if_silent": prior_if_silent,\n            "usable": bool(auc >= KILL_THRESHOLD) if np.isfinite(auc) else False,\n        })\n\n    return pd.DataFrame(records).set_index("finding")\n\n\ndef format_report(table: pd.DataFrame) -> str:\n    """Human-readable reliability table."""\n    lines = [\n        f"{\'finding\':18s} {\'n+\':>3s} {\'ment\':>5s} {\'prec\':>5s} {\'rec\':>5s} "\n        f"{\'AUC\':>5s} {\'P(1|silent)\':>11s} {\'use\':>4s}",\n        "-" * 66,\n    ]\n    for name, row in table.iterrows():\n        lines.append(\n            f"{name:18s} {int(row[\'n_positive\']):3d} "\n            f"{row[\'mention_rate\']:4.0%} {row[\'precision\']:5.2f} "\n            f"{row[\'recall\']:5.2f} {row[\'auc\']:5.3f} "\n            f"{row[\'prior_if_silent\']:11.2f} {\'yes\' if row[\'usable\'] else \'NO\':>4s}"\n        )\n    lines.append("-" * 66)\n    lines.append(f"macro AUC: {table[\'auc\'].mean():.3f}   "\n                 f"usable findings: {int(table[\'usable\'].sum())}/{len(table)}")\n    return "\\n".join(lines)\n\n\ndef main():\n    from src.labels import lexicon\n\n    table = score_extractor(lexicon.extract_finding)\n    print(format_report(table))\n\n\nif __name__ == "__main__":\n    main()\n',
    'src/labels/fuse.py': '"""Combine the rule-based and LLM extractors into soft training targets.\n\nProduces one row per study with, for each finding, a target in [0, 1] and a\nsample weight. Neither is a hard label: the extractors disagree often enough,\nand are wrong often enough, that training on thresholded output would teach the\nmodel to reproduce extraction errors.\n\n**Nothing here is fitted on the 58 gold studies.** An earlier version derived\nper-finding blend weights from gold AUCs and per-finding silence priors from\ngold positive rates -- 24 numbers estimated from 58 samples. It scored macro AUC\n0.860 in-sample and 0.812 under leave-one-out, which is worse than the LLM\nextractor alone at 0.838. The apparent gain was the fit, not the fusion.\n\nWhat replaced it is deliberately parameter-free:\n\n**A fixed blend.** Targets are 0.10 x lexicon + 0.90 x LLM for every finding.\nThe ratio sits on a plateau, not an argmax: with the dual-mode LLM scores,\n0.00 through 0.15 all land within 0.002 on gold (0.889-0.891) and the weight\nfalls away steadily above that -- the 0.30 chosen for the categorical-only\nLLM cost ~0.010 against the stronger extractor. A small nonzero weight keeps\nthe lexicon\'s independent evidence without letting it drag the ranking.\n\n**Silence scores zero, and is handled by the weight instead.** Measuring\nP(positive | both silent) on gold and using it as the target was worse than\nusing zero, on every variant tested. The concern that motivated it -- that a\nsilent finding is not a confident negative -- is real, and is addressed by\ngiving those studies a low sample weight rather than a fabricated target.\n\n**Disagreement lowers the weight rather than picking a winner.** A study where\nthe extractors conflict is genuinely ambiguous and should not pull as hard as\none where they concur.\n"""\n\nfrom pathlib import Path\nfrom typing import Optional\n\nimport numpy as np\nimport pandas as pd\n\nfrom src import config\nfrom src.labels import calibrate, lexicon\n\n# Blend ratio. Chosen from a plateau, not an argmax -- see module docstring.\nLEXICON_WEIGHT = 0.10\nLLM_WEIGHT = 1.0 - LEXICON_WEIGHT\n\n# Sample weights by how much the two extractors corroborate each other.\nWEIGHT_AGREE = 1.0        # both informative, same call\nWEIGHT_DISAGREE = 0.4     # both informative, opposite calls\nWEIGHT_SINGLE = 0.7       # one informative, one silent\nWEIGHT_SILENT = 0.25      # neither informative; target is zero but barely counts\n\n\ndef silence_rates(labels: pd.DataFrame) -> pd.Series:\n    """Per finding, the share of studies neither extractor said anything about.\n\n    Measured on the mined corpus -- 4,349 studies -- and never on gold. That\n    distinction is what separates this from the per-finding silence priors this\n    module removed: those were 24 numbers fitted on 58 samples and scored 0.860\n    in-sample against 0.812 leave-one-out. A rate over 4,349 studies is an\n    observation, not a fit.\n    """\n    return pd.Series(\n        {name: float((labels[f"{name}_weight"] == WEIGHT_SILENT).mean())\n         for name in config.LABELS})\n\n\ndef rescale_silent_weights(labels: pd.DataFrame) -> pd.DataFrame:\n    """Down-weight silence in proportion to how uninformative it is.\n\n    **The problem, measured on Synovitis.** 82.2% of studies are silent about\n    it, the mined target averages 0.087, and the true prevalence on gold is\n    46.6%. So roughly 1,666 of the 3,576 silent studies are true positives\n    being trained toward zero -- and because silence is so common here, those\n    studies carry 57% of the finding\'s gradient. The model is being taught to\n    rank real positives low. Synovitis duly scores 0.682 on gold, the only\n    finding where the fine-tuned model is WORSE than the retired frozen one\n    (0.746), which is what fitting label noise harder looks like.\n\n    **The fix, and why it is not the one that failed.** Targets are untouched:\n    fabricating P(positive | silent) as a target measured worse on every\n    variant tested, and that stands. What changes is the *weight*, which this\n    module\'s own docstring already names as the mechanism for silence being an\n    unconfident negative. The weight becomes\n\n        WEIGHT_SILENT x (1 - silence_rate(finding))\n\n    which is one rule with no free parameters, evaluated on the mined corpus.\n    Silence about ACL (9.2% of studies) stays worth nearly the full 0.25,\n    because a report that discusses ligaments and omits the ACL is saying\n    something. Silence about Synovitis (82.2%) falls to 0.044, because a\n    corpus that mentions it in one study out of six is not saying anything.\n\n    Returns a copy; the target columns are passed through unchanged.\n    """\n    rates = silence_rates(labels)\n    out = labels.copy()\n    for name in config.LABELS:\n        column = f"{name}_weight"\n        silent = out[column] == WEIGHT_SILENT\n        out.loc[silent, column] = WEIGHT_SILENT * (1.0 - rates[name])\n    return out\n\n\ndef _informative(present) -> bool:\n    """False when an extractor said nothing about this finding."""\n    if present is None:\n        return False\n    if isinstance(present, float) and np.isnan(present):\n        return False\n    return True\n\n\ndef _score(present, severity) -> float:\n    """One extractor\'s contribution to the target, in [0, 1]."""\n    if not _informative(present):\n        return 0.0\n    if int(present) == 1:\n        return float(severity) if severity == severity and severity is not None else 1.0\n    return 0.0\n\n\ndef lexicon_frame(reports, ids) -> pd.DataFrame:\n    """Run the rule-based extractor over a set of reports."""\n    records = []\n    for report in reports:\n        extracted = lexicon.extract_report(report)\n        row = {}\n        for name in config.LABELS:\n            row[name] = extracted[name]["present"]\n            row[f"{name}_severity"] = extracted[name]["severity"]\n        records.append(row)\n\n    frame = pd.DataFrame(records)\n    frame.insert(0, config.ID_COL, list(ids))\n    return frame\n\n\ndef fuse(lex: pd.DataFrame, llm: pd.DataFrame) -> pd.DataFrame:\n    """Blend two extractor frames into soft targets and sample weights."""\n    merged = lex.merge(llm, on=config.ID_COL, suffixes=("_lex", "_llm"))\n    if len(merged) != len(lex):\n        raise ValueError(f"merge lost rows: {len(lex)} -> {len(merged)}")\n\n    output = {config.ID_COL: merged[config.ID_COL].to_numpy()}\n\n    for name in config.LABELS:\n        lex_present = merged[f"{name}_lex"].to_numpy(dtype=object)\n        llm_present = merged[f"{name}_llm"].to_numpy(dtype=object)\n        lex_sev = merged[f"{name}_severity_lex"].to_numpy()\n        llm_sev = merged[f"{name}_severity_llm"].to_numpy()\n\n        targets = np.empty(len(merged))\n        sample_weights = np.empty(len(merged))\n\n        for i in range(len(merged)):\n            lex_score = _score(lex_present[i], lex_sev[i])\n            llm_score = _score(llm_present[i], llm_sev[i])\n            targets[i] = LEXICON_WEIGHT * lex_score + LLM_WEIGHT * llm_score\n\n            lex_ok = _informative(lex_present[i])\n            llm_ok = _informative(llm_present[i])\n            if lex_ok and llm_ok:\n                agree = (lex_score > 0) == (llm_score > 0)\n                sample_weights[i] = WEIGHT_AGREE if agree else WEIGHT_DISAGREE\n            elif lex_ok or llm_ok:\n                sample_weights[i] = WEIGHT_SINGLE\n            else:\n                sample_weights[i] = WEIGHT_SILENT\n\n        output[name] = np.clip(targets, 0.0, 1.0)\n        output[f"{name}_weight"] = sample_weights\n\n    return pd.DataFrame(output)\n\n\ndef fuse_scores(lex: pd.DataFrame, llm_scores: pd.DataFrame) -> pd.DataFrame:\n    """Blend the lexicon frame with an already-combined LLM score frame.\n\n    `llm_scores` is the output of `mode_fusion.combine` (or\n    `mode_fusion.from_categorical`): per finding a score in [0, 1], an\n    `_informative` flag, and a `_call` boolean. The weight semantics mirror\n    `fuse`: agreement compares the lexicon\'s call with the LLM\'s call.\n    """\n    merged = lex.merge(llm_scores, on=config.ID_COL, suffixes=("_lex", ""))\n    if len(merged) != len(lex):\n        raise ValueError(f"merge lost rows: {len(lex)} -> {len(merged)}")\n\n    output = {config.ID_COL: merged[config.ID_COL].to_numpy()}\n\n    for name in config.LABELS:\n        lex_present = merged[f"{name}_lex"].to_numpy(dtype=object)\n        lex_sev = merged[f"{name}_severity"].to_numpy()\n        llm_score = merged[name].to_numpy(dtype=float)\n        llm_ok = merged[f"{name}_informative"].to_numpy(dtype=bool)\n        llm_call = merged[f"{name}_call"].to_numpy(dtype=bool)\n\n        targets = np.empty(len(merged))\n        sample_weights = np.empty(len(merged))\n\n        for i in range(len(merged)):\n            lex_score = _score(lex_present[i], lex_sev[i])\n            targets[i] = LEXICON_WEIGHT * lex_score + LLM_WEIGHT * llm_score[i]\n\n            lex_ok = _informative(lex_present[i])\n            if lex_ok and llm_ok[i]:\n                agree = (lex_score > 0) == llm_call[i]\n                sample_weights[i] = WEIGHT_AGREE if agree else WEIGHT_DISAGREE\n            elif lex_ok or llm_ok[i]:\n                sample_weights[i] = WEIGHT_SINGLE\n            else:\n                sample_weights[i] = WEIGHT_SILENT\n\n        output[name] = np.clip(targets, 0.0, 1.0)\n        output[f"{name}_weight"] = sample_weights\n\n    return pd.DataFrame(output)\n\n\n# Order-keeper inside the reference distribution\'s tie blocks. Small enough\n# that the restored shape dominates the target value; the epsilon term only\n# arranges studies within a block.\nSHARPEN_EPSILON = 0.02\n\n\ndef sharpen_targets(fused: pd.DataFrame, reference: pd.DataFrame) -> pd.DataFrame:\n    """Restore `reference`\'s target distribution, keeping `fused`\'s ordering.\n\n    The dual-mode targets rank studies better than the categorical fusion\n    (0.891 vs 0.854 on gold) but drained the confident mass out of the\n    distribution -- extreme targets fell from 73% to 60% of slots and the\n    model trained on them regressed on the leaderboard (0.845 -> 0.827).\n    AUC is invariant under strictly monotone maps, so the ordering gain and\n    the crisp shape are separable: each finding\'s targets are quantile-mapped\n    onto the reference marginal, plus an epsilon-scaled rank term that keeps\n    the map strictly increasing where the reference has ties. Sample weights\n    are not touched.\n    """\n    from src.labels import mode_fusion\n\n    out = fused.copy()\n    for name in config.LABELS:\n        targets = fused[name].to_numpy(dtype=float)\n        ref = np.sort(reference[name].to_numpy(dtype=float))\n        ranks = mode_fusion._tied_ranks_01(targets)\n        mapped = np.quantile(ref, np.clip(ranks, 0.0, 1.0))\n        out[name] = np.clip(\n            (mapped + SHARPEN_EPSILON * ranks) / (1.0 + SHARPEN_EPSILON),\n            0.0, 1.0)\n    return out\n\n\ndef extractor_aucs(gold: pd.DataFrame, lex_gold: pd.DataFrame,\n                   llm_gold: pd.DataFrame) -> pd.DataFrame:\n    """Diagnostic only: how each extractor scores on gold.\n\n    These numbers are reported, never fed back into the targets. Letting them\n    set blend weights is exactly the overfit this module was rewritten to avoid.\n    """\n    rows = []\n    for name in config.LABELS:\n        truth = gold[name].to_numpy().astype(int)\n        lex_scores = np.array([\n            _score(p, s) for p, s in\n            zip(lex_gold[name], lex_gold[f"{name}_severity"])])\n        llm_scores = np.array([\n            _score(p, s) for p, s in\n            zip(llm_gold[name], llm_gold[f"{name}_severity"])])\n        rows.append({\n            "finding": name,\n            "lexicon_auc": calibrate._auc(truth, lex_scores),\n            "llm_auc": calibrate._auc(truth, llm_scores),\n        })\n    return pd.DataFrame(rows).set_index("finding")\n\n\ndef build(train_csv=None, llm_mined=None, llm_gold=None,\n          probs_mined=None, probs_gold=None,\n          out_path: Optional[str] = None):\n    """Full pipeline: extract, fuse, and score against gold.\n\n    When the probability-mode parquets exist alongside the categorical ones,\n    the two modes are combined by `mode_fusion` before the lexicon blend.\n    Otherwise this is the original categorical-only fusion, so the pipeline\n    still runs on a machine that never downloaded the probability dumps.\n    """\n    from src.labels import mode_fusion\n\n    gold = calibrate.load_gold(train_csv)\n    unlabelled = calibrate.load_unlabelled(train_csv)\n\n    mined_frame = pd.read_parquet(\n        llm_mined or (config.META_DIR / "llm_labels.parquet"))\n    gold_frame = pd.read_parquet(\n        llm_gold or (config.META_DIR / "llm_labels_gold.parquet"))\n\n    # Align to CSV order so positional comparisons stay valid.\n    gold_frame = (gold_frame.set_index(config.ID_COL)\n                  .loc[gold[config.ID_COL]].reset_index())\n    mined_frame = (mined_frame.set_index(config.ID_COL)\n                   .loc[unlabelled[config.ID_COL]].reset_index())\n\n    probs_mined_path = probs_mined or (config.META_DIR / "llm_probs.parquet")\n    probs_gold_path = probs_gold or (config.META_DIR / "llm_probs_gold.parquet")\n    use_probs = (Path(probs_mined_path).exists()\n                 and Path(probs_gold_path).exists())\n\n    if use_probs:\n        probs_mined_frame = pd.read_parquet(probs_mined_path)\n        probs_gold_frame = pd.read_parquet(probs_gold_path)\n        scores_mined = mode_fusion.combine(mined_frame, probs_mined_frame)\n        scores_gold = mode_fusion.combine(gold_frame, probs_gold_frame)\n    else:\n        scores_mined = mode_fusion.from_categorical(mined_frame)\n        scores_gold = mode_fusion.from_categorical(gold_frame)\n\n    lex_gold = lexicon_frame(gold["Report"], gold[config.ID_COL])\n    lex_mined = lexicon_frame(unlabelled["Report"], unlabelled[config.ID_COL])\n\n    fused = fuse_scores(lex_mined, scores_mined)\n    fused_gold = fuse_scores(lex_gold, scores_gold)\n\n    if use_probs:\n        # The categorical-only fusion of the same inputs supplies the shape\n        # the dual-mode targets are sharpened back onto.\n        ref_mined = fuse_scores(\n            lex_mined, mode_fusion.from_categorical(mined_frame))\n        ref_gold = fuse_scores(\n            lex_gold, mode_fusion.from_categorical(gold_frame))\n        fused = sharpen_targets(fused, ref_mined)\n        fused_gold = sharpen_targets(fused_gold, ref_gold)\n\n    aucs = extractor_aucs(gold, lex_gold, gold_frame)\n\n    if out_path:\n        fused.to_parquet(out_path, index=False)\n\n    return fused, fused_gold, aucs\n\n\ndef report(gold: pd.DataFrame, fused_gold: pd.DataFrame,\n           aucs: pd.DataFrame, fused: pd.DataFrame) -> str:\n    """Per-finding summary of the fused targets.\n\n    The fused AUC column is an honest estimate: no parameter in this module was\n    estimated from the gold studies, so in-sample and held-out are the same.\n    """\n    lines = [\n        f"{\'finding\':18s} {\'lexAUC\':>7s} {\'llmAUC\':>7s} {\'fusedAUC\':>9s} "\n        f"{\'delta\':>7s} {\'mean w\':>7s} {\'silent\':>7s}",\n        "-" * 72,\n    ]\n    fused_aucs = []\n    for name in config.LABELS:\n        truth = gold[name].to_numpy().astype(int)\n        auc = calibrate._auc(truth, fused_gold[name].to_numpy())\n        fused_aucs.append(auc)\n        best_single = max(aucs.loc[name, "lexicon_auc"], aucs.loc[name, "llm_auc"])\n        silent = float((fused[f"{name}_weight"] == WEIGHT_SILENT).mean())\n        lines.append(\n            f"{name:18s} {aucs.loc[name, \'lexicon_auc\']:7.3f} "\n            f"{aucs.loc[name, \'llm_auc\']:7.3f} {auc:9.3f} "\n            f"{auc - best_single:+7.3f} "\n            f"{fused[f\'{name}_weight\'].mean():7.2f} {silent:6.0%}"\n        )\n    lines.append("-" * 72)\n    lines.append(\n        f"macro: fused {np.nanmean(fused_aucs):.3f}   "\n        f"lexicon {aucs[\'lexicon_auc\'].mean():.3f}   "\n        f"LLM {aucs[\'llm_auc\'].mean():.3f}"\n    )\n    return "\\n".join(lines)\n\n\ndef main():\n    out = config.META_DIR / "train_labels.parquet"\n    fused, fused_gold, aucs = build(out_path=out)\n    gold = calibrate.load_gold()\n    print(report(gold, fused_gold, aucs, fused))\n    print(f"\\nwrote {out}  {fused.shape}")\n\n\nif __name__ == "__main__":\n    main()\n',
    'src/labels/mode_fusion.py': '"""Combine the categorical and probability extraction modes into one score.\n\nThe two LLM extraction modes fail in complementary ways. The categorical mode\nis precise where reports state a finding crisply (Medial Meniscus 0.944, MCL\n0.959 on gold) but collapses more than half of all label slots onto a single\nvalue, and AUC cannot order studies inside a tie block. The probability mode\nspreads the silent studies (Fracture 0.885 vs 0.752) but leaves Medial\nMeniscus and MCL nearly all tied. Best-of-both per finding scores ~0.869\nagainst 0.854 for the categorical-only fusion -- but picking a winner per\nfinding on 58 gold studies is the same overfit that sank the fitted fusion\nweights (0.860 in-sample, 0.812 leave-one-out).\n\nEvery strategy here therefore applies one rule globally, with at most one\nglobal parameter, so in-sample and held-out are the same number. Per-finding\nselection is deliberately not offered.\n\nMeasured on the 32B gold dump (2026-08-23): categorical alone 0.838,\nprobability alone 0.860, `score_average` and `rank_average` tied at 0.889 --\nabove the 0.885 per-finding oracle, because averaging complementary errors\nbeats choosing between them. The delete-8 jackknife split 0.54/0.47 between\nthe two, which is a tie, and the tie-break rule is simplicity: freeze\n`score_average`. It also keeps a probability-like scale (mean target 0.33\nagainst gold prevalence 0.35, where rank scores are uniform at 0.5 by\nconstruction), which matters because these become BCE targets downstream --\nimputing silence priors as targets measured worse than zero every time.\n"""\n\nfrom typing import Optional\n\nimport numpy as np\nimport pandas as pd\n\nfrom src import config\nfrom src.labels import fuse\n\n# Frozen from the gold-dump comparison recorded in the module docstring.\nDEFAULT_STRATEGY = "score_average"\n\n# Tiebreak weight. Correctness only needs it below the smallest gap between\n# distinct categorical scores (severity x certainty products; adjacent values\n# sit ~0.05 apart), so 0.01 is not a tuned quantity.\nTIEBREAK_EPSILON = 0.01\n\n# Global blend weight on the categorical rank for the `global_blend` strategy.\n# Must be set from a plateau, never an argmax -- see eval_fusion.py.\nGLOBAL_BLEND_ALPHA = 0.5\n\nSTRATEGIES = (\n    "score_average",\n    "rank_average",\n    "cat_primary_tiebreak",\n    "global_blend",\n    "confidence_weighted",\n)\n\n\ndef _tied_ranks_01(values: np.ndarray) -> np.ndarray:\n    """Tie-averaged ranks rescaled to [0, 1]; NaN positions come back NaN."""\n    result = np.full(len(values), np.nan)\n    mask = ~np.isnan(values)\n    n = int(mask.sum())\n    if n == 0:\n        return result\n    if n == 1:\n        result[mask] = 0.5\n        return result\n\n    subset = values[mask]\n    order = np.argsort(subset, kind="mergesort")\n    ranks = np.empty(n, dtype=float)\n    ranks[order] = np.arange(1, n + 1)\n    unique, inverse, counts = np.unique(subset, return_inverse=True,\n                                        return_counts=True)\n    tie_sums = np.zeros(len(unique))\n    np.add.at(tie_sums, inverse, ranks)\n    ranks = (tie_sums / counts)[inverse]\n\n    result[mask] = (ranks - 1.0) / (n - 1.0)\n    return result\n\n\ndef _categorical_scores(cat_frame: pd.DataFrame, name: str) -> np.ndarray:\n    """Map one finding\'s categorical columns through the fusion score."""\n    return np.array([\n        fuse._score(p, s) for p, s in\n        zip(cat_frame[name], cat_frame[f"{name}_severity"])\n    ])\n\n\ndef _categorical_informative(cat_frame: pd.DataFrame, name: str) -> np.ndarray:\n    return np.array([fuse._informative(p) for p in cat_frame[name]])\n\n\ndef _combine_finding(cat_score, cat_ok, prob, prob_conf, cat_conf,\n                     strategy: str) -> np.ndarray:\n    """Combine one finding\'s two score vectors. `prob` may contain NaN."""\n    has_prob = ~np.isnan(prob)\n\n    if strategy == "score_average":\n        # Plain mean of the two scores. On the gold dump this matched rank\n        # averaging\'s AUC exactly while keeping a probability-like scale --\n        # its mean target sits at the gold prevalence, where rank scores are\n        # uniform by construction. Scale matters because these become BCE\n        # targets downstream.\n        return np.where(has_prob, 0.5 * (cat_score + prob), cat_score)\n\n    if strategy == "cat_primary_tiebreak":\n        tiebreak = np.where(has_prob, prob, 0.0)\n        combined = cat_score + TIEBREAK_EPSILON * tiebreak\n        return combined / (1.0 + TIEBREAK_EPSILON)\n\n    if strategy == "rank_average":\n        cat_rank = _tied_ranks_01(cat_score)\n        prob_rank = _tied_ranks_01(np.where(has_prob, prob, np.nan))\n        return np.where(has_prob, 0.5 * (cat_rank + prob_rank), cat_rank)\n\n    if strategy == "global_blend":\n        alpha = GLOBAL_BLEND_ALPHA\n        cat_rank = _tied_ranks_01(cat_score)\n        prob_rank = _tied_ranks_01(np.where(has_prob, prob, np.nan))\n        return np.where(has_prob,\n                        alpha * cat_rank + (1.0 - alpha) * prob_rank,\n                        cat_rank)\n\n    if strategy == "confidence_weighted":\n        w_cat = np.where(cat_ok, np.nan_to_num(cat_conf, nan=0.0), 0.0)\n        w_prob = np.where(has_prob, np.nan_to_num(prob_conf, nan=0.0), 0.0)\n        total = w_cat + w_prob\n        # Neither mode confident: fall back to the categorical score, which\n        # is zero for silence anyway.\n        blended = np.where(\n            total > 0,\n            (w_cat * cat_score + w_prob * np.where(has_prob, prob, 0.0))\n            / np.where(total > 0, total, 1.0),\n            cat_score,\n        )\n        return blended\n\n    raise ValueError(f"unknown strategy {strategy!r}; one of {STRATEGIES}")\n\n\ndef combine(cat_frame: pd.DataFrame, prob_frame: pd.DataFrame,\n            strategy: str = DEFAULT_STRATEGY) -> pd.DataFrame:\n    """Merge the two extraction modes into one score frame.\n\n    Output has, per finding: `<name>` (score in [0, 1]), `<name>_informative`\n    (the *categorical* mode asserted something -- see the comment below, this\n    deliberately excludes the probability mode\'s context-guesses), and\n    `<name>_call` (the positive/negative call used for agreement weighting\n    downstream -- the categorical call where it exists, else probability\n    >= 0.5).\n    """\n    merged = cat_frame.merge(prob_frame, on=config.ID_COL,\n                             suffixes=("", "_prob"))\n    if len(merged) != len(cat_frame):\n        raise ValueError(\n            f"mode merge lost rows: {len(cat_frame)} -> {len(merged)}")\n\n    output = {config.ID_COL: merged[config.ID_COL].to_numpy()}\n    for name in config.LABELS:\n        cat_score = _categorical_scores(merged, name)\n        cat_ok = _categorical_informative(merged, name)\n        cat_conf = merged.get(\n            f"{name}_confidence",\n            pd.Series(1.0, index=merged.index)).to_numpy(dtype=float)\n        prob = merged[f"{name}_prob"].to_numpy(dtype=float)\n        prob_conf_col = merged.get(f"{name}_confidence_prob")\n        prob_conf = (prob_conf_col.to_numpy(dtype=float)\n                     if prob_conf_col is not None\n                     else np.ones(len(merged)))\n        has_prob = ~np.isnan(prob)\n\n        combined = _combine_finding(cat_score, cat_ok, prob, prob_conf,\n                                    cat_conf, strategy)\n        output[name] = np.clip(combined, 0.0, 1.0)\n        # Informative means an extractor *asserted* something, which only the\n        # categorical mode tracks. The probability mode is never silent -- its\n        # prompt asks it to estimate unmentioned findings from context -- so\n        # `cat_ok | has_prob` would be True everywhere and would silently\n        # retire the silence down-weighting. That is not a free change: 34.5%\n        # of slots are both-silent, and lifting them from WEIGHT_SILENT to\n        # WEIGHT_SINGLE puts 2.8x the training mass on the least trustworthy\n        # third of the corpus. The probability estimate still ranks studies\n        # inside that block, which is the tie-breaking gain worth having --\n        # the value is kept, the certainty is not asserted.\n        output[f"{name}_informative"] = cat_ok\n        output[f"{name}_call"] = np.where(\n            cat_ok, cat_score > 0, np.where(has_prob, prob >= 0.5, False)\n        ).astype(bool)\n\n    return pd.DataFrame(output)\n\n\ndef from_categorical(cat_frame: pd.DataFrame) -> pd.DataFrame:\n    """A score frame from the categorical mode alone.\n\n    The baseline every strategy is compared against, and the shape `combine`\n    degrades to when the probability frame has nothing to say.\n    """\n    output = {config.ID_COL: cat_frame[config.ID_COL].to_numpy()}\n    for name in config.LABELS:\n        score = _categorical_scores(cat_frame, name)\n        ok = _categorical_informative(cat_frame, name)\n        output[name] = score\n        output[f"{name}_informative"] = ok\n        output[f"{name}_call"] = (score > 0) & ok\n    return pd.DataFrame(output)\n',
    'tools/audit_labels.py': '"""Audit of the label tributary: reproduction, properties, and language reach.\n\n`train_labels.parquet` is the other input to every fold the 0.906 ensemble is\nmade of, and step 1 verified only the image side. This does for the label\nmatrix what `verify_cache.py` did for the cache -- rebuild it from the code\nthat claims to produce it and compare -- and then asks the question step 2\nasked of the side classifier: what population is the reported accuracy\nmeasured on, versus the population the artifact is applied to?\n\nFour parts, each independently useful:\n\n  A. REPRODUCTION.  Rebuild the shipped matrix from `fuse.build()` and the\n     extractor dumps, and compare every cell. Cheap enough (~10 s) to cover\n     100% of the corpus rather than the 2% sample `verify_cache.py` could\n     afford, because nothing here decodes a pixel.\n\n  B. PROPERTIES.  The numbers the docs quote about the artifact -- 75% extreme\n     targets, mean weight 0.625, 34.5% at WEIGHT_SILENT -- read off the file\n     rather than off the worklog. Step 2 found the side classifier\'s\n     documented accuracy belonged to a run that no longer existed; this is the\n     same check for the labels.\n\n  D. AGAINST THE GOLD IMAGES.  Miss rate and false-alarm rate against the 58\n     radiologist-labelled studies -- the only external truth there is. Kept\n     apart from the AUC, which measures ordering and cannot see that these\n     labels are precise and incomplete.\n\n  C. LANGUAGE REACH.  The corpus is multilingual and the lexicon is one of the\n     two signals whose *informativeness* sets every sample weight. If its\n     reach varies by language, then studies are weighted by what language they\n     were written in, which is not a property anybody chose. Reported as a\n     length-matched comparison, because non-English reports are also shorter\n     and the confound would otherwise carry the result.\n\n    python tools/audit_labels.py\n    python tools/audit_labels.py --sweep      # also sweep LEXICON_WEIGHT\n\nExit status is non-zero if the shipped matrix does not reproduce, or if a\ndocumented property has drifted. The language section never fails the run --\nit is a measurement, not a defect, and what to do about it costs five fold\nretrains.\n"""\n\nimport argparse\nimport re\nimport sys\nimport unicodedata\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\n\nsys.path.insert(0, str(Path(__file__).resolve().parent.parent))\n\nfrom src import config\nfrom src.labels import calibrate, fuse, mode_fusion\n\n# ---------------------------------------------------------------------------\n# Language cues\n# ---------------------------------------------------------------------------\n# No language detector is installed and none is worth a dependency for this:\n# every report in this corpus names the knee, the menisci and a side, so\n# anatomy and laterality words identify the language with far higher precision\n# than general-purpose stopwords would. Anything short of a clear margin is\n# left "unknown" rather than guessed -- an audit that mislabels its own strata\n# is worse than one that admits a gap.\nCUES = {\n    "en": ["knee", "meniscus", "medial", "lateral", "tear", "effusion",\n           "the ", "and ", "with", "joint"],\n    "es": ["rodilla", "menisco", "izquierd", "derech", "articulaci",\n           "tecnica", "los ", "del "],\n    "pt": ["joelho", "menisco", "direito", "esquerdo", "articula", "ossea"],\n    "it": ["ginocchio", "menisco", "destro", "sinistro", "articolazione",\n           "legamento"],\n    "fr": ["genou", "menisque", "droit", "gauche", "articulation",\n           "ligament croise"],\n    "de": ["knie", "meniskus", "rechts", "links", "gelenk", "kreuzband",\n           "erguss", "nachweis"],\n    "nl": ["knie", "meniscus", "rechter", "linker", "gewricht", "kruisband",\n           "het ", "van de"],\n    "tr": ["diz", "menisk", "sag ", "sol ", "eklem", "kemik", "bag ",\n           "izlenmi"],\n    "hr": ["koljen", "meniskus", "desno", "lijevo", "zglob", "ligament",\n           "prikaz", "promjen"],\n}\n_GREEK = re.compile(r"[Ͱ-Ͽἀ-\u1fff]")\n_CYRILLIC = re.compile(r"[Ѐ-ӿ]")\n\n\ndef detect_language(text) -> str:\n    """One of the CUES keys, a script code, \'unknown\', or \'empty\'."""\n    if not isinstance(text, str) or not text.strip():\n        return "empty"\n    lowered = fuse.lexicon._norm(text)\n    length = len(lowered)\n\n    if len(_GREEK.findall(lowered)) > 0.10 * length:\n        return "el"\n    if len(_CYRILLIC.findall(lowered)) > 0.10 * length:\n        # Serbian Cyrillic carries ј љ њ ћ џ; Bulgarian leans on щ ъ;\n        # Russian on ы э ё. One of the three is usually unambiguous.\n        counts = [(sum(lowered.count(c) for c in "јљњћџ"), "sr"),\n                  (sum(lowered.count(c) for c in "щъ"), "bg"),\n                  (sum(lowered.count(c) for c in "ыэё"), "ru")]\n        best = max(counts)\n        return best[1] if best[0] else "cyrillic?"\n\n    scores = sorted(((sum(lowered.count(c) for c in cues), name)\n                     for name, cues in CUES.items()), reverse=True)\n    top, second = scores[0], scores[1]\n    if top[0] < 2 or top[0] < second[0] * 1.5:\n        return "unknown"\n    return top[1]\n\n\n# ---------------------------------------------------------------------------\n# A. Reproduction\n# ---------------------------------------------------------------------------\n\ndef check_reproduction(shipped: pd.DataFrame, rebuilt: pd.DataFrame,\n                       header: str = "A. REPRODUCTION") -> bool:\n    """Every cell of the shipped matrix, against a rebuild from source."""\n    print(header)\n    ok = True\n    if list(shipped.columns) != list(rebuilt.columns):\n        print("   [FAIL] column set differs")\n        return False\n    if not shipped[config.ID_COL].equals(rebuilt[config.ID_COL]):\n        print("   [FAIL] study order differs")\n        return False\n\n    left = shipped.set_index(config.ID_COL)\n    right = rebuilt.set_index(config.ID_COL).loc[left.index]\n    worst = 0.0\n    differing = 0\n    for column in left.columns:\n        delta = np.abs(left[column].to_numpy(float)\n                       - right[column].to_numpy(float))\n        worst = max(worst, float(delta.max()))\n        differing += int((delta > 1e-12).sum())\n\n    cells = left.shape[0] * left.shape[1]\n    if differing:\n        print(f"   [FAIL] {differing:,} of {cells:,} cells differ, "\n              f"max |delta| {worst:.3e}")\n        ok = False\n    else:\n        print(f"   [PASS] {cells:,} cells identical across "\n              f"{len(left):,} studies and {len(left.columns)} columns")\n    return ok\n\n\n# ---------------------------------------------------------------------------\n# B. Documented properties\n# ---------------------------------------------------------------------------\n\nDOCUMENTED = {"extreme_share": 0.75, "mean_weight": 0.625, "silent_share": 0.345}\n\n\ndef check_properties(labels: pd.DataFrame, tolerance: float = 0.01,\n                     header: str = "B. DOCUMENTED PROPERTIES") -> bool:\n    """The numbers the docs quote, read off the artifact itself."""\n    print("\\n" + header)\n    targets = labels[config.LABELS].to_numpy(float)\n    weights = labels[[f"{c}_weight" for c in config.LABELS]].to_numpy(float)\n    measured = {\n        "extreme_share": float(np.mean((targets < 0.1) | (targets > 0.9))),\n        "mean_weight": float(weights.mean()),\n        "silent_share": float(np.mean(np.isclose(weights, fuse.WEIGHT_SILENT))),\n    }\n    ok = True\n    for name, expected in DOCUMENTED.items():\n        got = measured[name]\n        status = "PASS" if abs(got - expected) <= tolerance else "FAIL"\n        ok &= status == "PASS"\n        print(f"   [{status}] {name:14s} {got:.4f}   documented {expected}")\n    return ok\n\n\n# ---------------------------------------------------------------------------\n# C. Language reach\n# ---------------------------------------------------------------------------\n\ndef language_reach(reports: pd.DataFrame, unlabelled: pd.DataFrame,\n                   deciles: int = 10, header: str = "C. LANGUAGE REACH"):\n    """Per-language informativeness of each of the two extractor signals.\n\n    The weight of every slot is set by two booleans -- did the lexicon say\n    anything, did the LLM say anything -- so this is the quantity that decides\n    how much a study is allowed to teach, independent of what it says.\n    """\n    print("\\n" + header)\n    language = reports.set_index(config.ID_COL)["language"]\n    groups = unlabelled[config.ID_COL].map(language).to_numpy()\n\n    mined = (pd.read_parquet(config.META_DIR / "llm_labels.parquet")\n             .set_index(config.ID_COL)\n             .loc[unlabelled[config.ID_COL]].reset_index())\n    scores = mode_fusion.combine(\n        mined, pd.read_parquet(config.META_DIR / "llm_probs.parquet"))\n    lex = fuse.lexicon_frame(unlabelled["Report"], unlabelled[config.ID_COL])\n\n    lex_ok = np.column_stack([lex[c].map(fuse._informative).to_numpy()\n                              for c in config.LABELS])\n    llm_ok = np.column_stack([scores[f"{c}_informative"].to_numpy(bool)\n                              for c in config.LABELS])\n    chars = unlabelled["Report"].str.len().to_numpy()\n\n    rows = []\n    for name in pd.Series(groups).value_counts().index:\n        mask = groups == name\n        rows.append({\n            "lang": name,\n            "n": int(mask.sum()),\n            "lexicon %": 100 * lex_ok[mask].mean(),\n            "llm %": 100 * llm_ok[mask].mean(),\n            "both %": 100 * (lex_ok[mask] & llm_ok[mask]).mean(),\n            "median chars": int(np.median(chars[mask])),\n        })\n    table = pd.DataFrame(rows)\n    print("\\n   informative share of the 12 slots, per language\\n")\n    print(table.to_string(index=False, float_format=lambda x: f"{x:.1f}"))\n\n    # The confound: non-English reports are also shorter, and a shorter report\n    # genuinely mentions fewer findings. Comparing inside corpus-wide length\n    # deciles removes it.\n    bins = pd.qcut(chars, deciles, labels=False, duplicates="drop")\n    big = [row["lang"] for _, row in table.iterrows()\n           if row["n"] >= 80 and row["lang"] not in ("unknown", "empty")]\n    matched = pd.DataFrame(index=[f"d{i}" for i in range(deciles)])\n    for name in big:\n        mask = groups == name\n        matched[name] = [\n            100 * lex_ok[mask & (bins == b)].mean()\n            if (mask & (bins == b)).sum() >= 10 else np.nan\n            for b in range(deciles)]\n    print("\\n   length-matched: lexicon informative %, by report-length decile\\n")\n    print(matched.to_string(float_format=lambda x: f"{x:.1f}"))\n    return table\n\n\ndef stem_reach(reports: pd.DataFrame):\n    """Where a stem list misses a language, name the word it missed.\n\n    A statistical gap is an observation; the stem that does not fire is a\n    cause. Multi-word stems are the exposure -- they require the two words\n    adjacent and uninflected, which Latin word order gives for free and\n    Slavic definite-article suffixes and coordinated noun phrases do not.\n    """\n    print("\\n   stem lists that do not reach their own language")\n    print(f"   {\'finding\':18s} {\'stems\':>5s} {\'multi-word\':>11s}")\n    for name in ("Medial Meniscus", "Lateral Meniscus", "ACL"):\n        stems = fuse.lexicon.FINDINGS[name]["anatomy"]\n        multi = sum(" " in s for s in stems)\n        print(f"   {name:18s} {len(stems):5d} {multi:10d}")\n\n    print()\n    probes = [("bg", "мениск", "Medial Meniscus"),\n              ("hr", "menisk", "Medial Meniscus"),\n              ("el", "μηνισκ", "Medial Meniscus")]\n    for lang, probe, finding in probes:\n        subset = reports[reports["language"] == lang]\n        if subset.empty:\n            continue\n        text = subset["Report"].map(fuse.lexicon._norm)\n        pattern = fuse.lexicon._stem_pattern(\n            fuse.lexicon.FINDINGS[finding]["anatomy"])\n        bare = 100 * text.str.contains(probe, regex=False).mean()\n        hit = 100 * text.map(lambda t: bool(pattern.search(t))).mean()\n        print(f"   {lang}: reports containing \'{probe}\' {bare:5.1f}%   "\n              f"matched by the {finding} stems {hit:5.1f}%")\n\n\n# ---------------------------------------------------------------------------\n# F. Against the gold images\n# ---------------------------------------------------------------------------\n\nWEIGHT_MEANING = {\n    fuse.WEIGHT_AGREE: "both signals agree",\n    fuse.WEIGHT_SINGLE: "only one signal spoke",\n    fuse.WEIGHT_DISAGREE: "the two signals disagree",\n    fuse.WEIGHT_SILENT: "neither signal spoke",\n}\n\n\ndef check_against_gold(gold: pd.DataFrame, fused_gold: pd.DataFrame,\n                       threshold: float = 0.5,\n                       header: str = "F. AGAINST THE GOLD IMAGES") -> pd.DataFrame:\n    """Miss rate and false-alarm rate against radiologist image labels.\n\n    The AUC in Part D says how well the targets *order* studies. It cannot say\n    whether they are right, and the two come apart here: these labels are\n    precise and incomplete, because a report is not an inventory of what the\n    image contains. A finding the radiologist saw and did not write down is\n    absent from the report, and the pipeline can only write a zero.\n\n    So misses and false alarms are reported separately -- a pooled error rate\n    hides this entirely, since most slots are true negatives and calling them\n    all negative already scores well. Broken out by the pipeline\'s own weight,\n    which is the interesting cut: the weight is assigned before anything looks\n    at truth, so it either predicts error or it does not.\n    """\n    print("\\n" + header)\n    labels = config.LABELS\n    truth = np.column_stack([gold[n].to_numpy().astype(int) for n in labels])\n    score = np.column_stack([fused_gold[n].to_numpy(float) for n in labels])\n    weight = np.column_stack([fused_gold[f"{n}_weight"].to_numpy(float)\n                              for n in labels])\n    call = score > threshold\n\n    def rates(mask):\n        t, c = truth[mask], call[mask]\n        positive, negative = t == 1, t == 0\n        return (int(mask.sum()), float(positive.mean()) if mask.sum() else np.nan,\n                float((~c[positive]).mean()) if positive.sum() else np.nan,\n                float(c[negative].mean()) if negative.sum() else np.nan)\n\n    print(f"   {\'weight\':>7s} {\'meaning\':26s} {\'slots\':>6s} {\'truly +\':>8s}"\n          f" {\'missed\':>8s} {\'false +\':>8s}")\n    for value in sorted(WEIGHT_MEANING, reverse=True):\n        n, positive, miss, false_alarm = rates(np.isclose(weight, value))\n        if not n:\n            continue\n        print(f"   {value:7.2f} {WEIGHT_MEANING[value]:26s} {n:6d}"\n              f" {100 * positive:7.1f}% {100 * miss:7.1f}% {100 * false_alarm:7.1f}%")\n    n, positive, miss, false_alarm = rates(np.ones_like(call, dtype=bool))\n    print(f"   {\'\':7s} {\'ALL\':26s} {n:6d} {100 * positive:7.1f}%"\n          f" {100 * miss:7.1f}% {100 * false_alarm:7.1f}%")\n\n    rows = []\n    for i, name in enumerate(labels):\n        t, c = truth[:, i], call[:, i]\n        if t.sum() == 0:\n            continue\n        rows.append({"finding": name, "positives": int(t.sum()),\n                     "missed %": 100 * float((~c[t == 1]).mean()),\n                     "false + %": 100 * float(c[t == 0].mean())\n                     if (t == 0).sum() else np.nan})\n    columns = ["finding", "positives", "missed %", "false + %"]\n    # No positives anywhere means every miss rate is undefined rather than\n    # zero, so the table is empty by construction and must not be sorted into\n    # a KeyError. Reachable with a small or skewed gold set.\n    table = (pd.DataFrame(rows).sort_values("missed %", ascending=False)\n             if rows else pd.DataFrame(columns=columns))\n    print("\\n   per finding, worst first (n is small -- read the order, not"\n          " the decimals)\\n")\n    print(table.to_string(index=False, float_format=lambda x: f"{x:.1f}"))\n\n    # Silence is the design decision this measures, so it gets stated outright\n    # rather than left for the reader to derive from the table.\n    silent = np.isclose(weight, fuse.WEIGHT_SILENT)\n    hidden = int((silent & (truth == 1)).sum())\n    if silent.sum():\n        print(f"\\n   {hidden} of {int(silent.sum())} silent slots are truly"\n              f" positive and are written as zero by construction.")\n        print("   That is the cost of \'silence means absent, weighted low\' --"\n              " a design choice,")\n        print("   measured here rather than assumed.")\n    return table\n\n\ndef sweep_lexicon_weight(gold: pd.DataFrame):\n    """Gold macro against the lexicon\'s share of the target.\n\n    Two things are worth reading off this: whether the shipped 0.10 is on a\n    plateau, and whether the sample weights move at all. They do not -- weights\n    are set by informativeness, not by this constant -- so the target half and\n    the weight half of the lexicon\'s influence have to be decided separately.\n    """\n    print("\\nD. LEXICON_WEIGHT SWEEP (gold n=58, macro SE ~0.015-0.025)\\n")\n    original = fuse.LEXICON_WEIGHT\n    rows = []\n    try:\n        for weight in (0.00, 0.05, 0.10, 0.15, 0.20, 0.30):\n            fuse.LEXICON_WEIGHT = weight\n            fuse.LLM_WEIGHT = 1.0 - weight\n            fused, fused_gold, _ = fuse.build()\n            aucs = [calibrate._auc(gold[n].to_numpy().astype(int),\n                                   fused_gold[n].to_numpy())\n                    for n in config.LABELS]\n            weights = fused[[f"{c}_weight" for c in config.LABELS]].to_numpy(float)\n            rows.append({"LEXICON_WEIGHT": weight,\n                         "gold macro": float(np.nanmean(aucs)),\n                         "mean weight": float(weights.mean())})\n    finally:\n        fuse.LEXICON_WEIGHT = original\n        fuse.LLM_WEIGHT = 1.0 - original\n    table = pd.DataFrame(rows)\n    print(table.to_string(index=False, float_format=lambda x: f"{x:.4f}"))\n    if table["mean weight"].nunique() == 1:\n        print("\\n   sample weights are identical across the sweep: this knob "\n              "moves targets only.")\n    return table\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--labels", default=None,\n                        help="shipped train_labels.parquet; defaults to META_DIR")\n    parser.add_argument("--sweep", action="store_true",\n                        help="also sweep LEXICON_WEIGHT against gold")\n    parser.add_argument("--out", default=None,\n                        help="write the per-language table to this csv")\n    args = parser.parse_args()\n\n    shipped_path = Path(args.labels or (config.META_DIR / "train_labels.parquet"))\n    shipped = pd.read_parquet(shipped_path)\n    print(f"shipped  {shipped_path}  {shipped.shape}\\n")\n\n    rebuilt, fused_gold, _ = fuse.build()\n    ok = check_reproduction(shipped, rebuilt)\n    ok &= check_properties(shipped)\n    check_against_gold(calibrate.load_gold(config.TRAIN_CSV), fused_gold)\n\n    reports = pd.read_csv(config.TRAIN_CSV)\n    reports["language"] = reports["Report"].map(detect_language)\n    unlabelled = calibrate.load_unlabelled(config.TRAIN_CSV)\n    table = language_reach(reports, unlabelled)\n    stem_reach(reports)\n\n    if args.sweep:\n        sweep_lexicon_weight(calibrate.load_gold(config.TRAIN_CSV))\n    if args.out:\n        table.to_csv(args.out, index=False)\n        print(f"\\nwrote {args.out}")\n\n    print("\\n" + ("ALL CHECKS PASSED" if ok else "CHECKS FAILED"))\n    return 0 if ok else 1\n\n\nif __name__ == "__main__":\n    sys.exit(main())\n'
}

for path, source in FILES.items():
    target = pathlib.Path(path)
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_text(source)

sys.path.insert(0, ".")

# Drop any previously imported copy before importing. Writing the files above
# does not affect a module already in sys.modules, so re-running this cell in a
# kernel that has already imported src/ silently keeps the OLD code -- the new
# file sits on disk unused, and the run looks fine until something calls a
# function that only exists in the new version.
for _name in [m for m in sys.modules if m == "src" or m.startswith("src.")]:
    del sys.modules[_name]
importlib.invalidate_caches()

from src import config
from src.data import dicom_io, cache, index as index_mod

# Only some notebooks materialise the model package.
if pathlib.Path("src/model/side_classifier.py").exists():
    from src.model import side_classifier
if pathlib.Path("src/model/net.py").exists():
    from src.model import (augment as knee_augment, averaging, conditional,
                           embed, dataset as knee_dataset, net, splits)
    from src.model import artifacts, experiments
# Only the fine-tuning notebook materialises these; lightning imports its
# framework at module scope and fuse pulls in the two extractors.
if pathlib.Path("src/model/inspect.py").exists():
    from src.model import inspect as knee_inspect
if pathlib.Path("src/labels/fuse.py").exists():
    from src.labels import fuse
if pathlib.Path("src/model/lightning.py").exists():
    from src.model import lightning as knee_lightning

print("modules ready:", len(FILES), "files")
print("labels:", len(config.LABELS))
print("cache geometry:", config.CACHE_SLICES, "x", config.CACHE_SIZE)
print("data root:", config.DATA_ROOT)

## What is actually attached

Printed before anything runs. The commonest failure in this project is a
missing or stale input rather than a real defect, and unlike a missing cache
shard, a stale label file produces no symptom at all — the run trains happily
on the wrong targets.

`config.META_DIR` is rebound to a staging directory here. The label modules
resolve their inputs through it at call time, so pointing it at the attached
copies is what lets `fuse.build()` run unmodified rather than reimplemented —
the same rule notebook 12 follows for the cache.

In [ ]:
import glob
import hashlib
import os
import shutil
from pathlib import Path

import numpy as np
import pandas as pd

from src import config
from src.labels import calibrate, fuse, mode_fusion


def find_inputs(pattern):
    # The competition directory is skipped: ~820,000 DICOM files and never
    # where these live.
    found = sorted(glob.glob(f"/kaggle/working/{pattern}"))
    for root in sorted(glob.glob("/kaggle/input/*")):
        if os.path.basename(root) == "competitions":
            continue
        found += sorted(glob.glob(f"{root}/**/{pattern}", recursive=True))
    return found


def find_competition_file(name):
    for candidate in (f"data/meta/{name}", f"/kaggle/input/competitions/*/{name}",
                      f"/kaggle/input/*/{name}"):
        hits = sorted(glob.glob(candidate))
        if hits:
            return hits[0]
    return None


STAGE = Path("/kaggle/working/meta")
STAGE.mkdir(parents=True, exist_ok=True)

WANTED = ["train_labels.parquet", "llm_labels.parquet", "llm_labels_gold.parquet",
          "llm_probs.parquet", "llm_probs_gold.parquet"]
staged = {}
for name in WANTED:
    hits = find_inputs(name)
    if hits:
        shutil.copy(hits[0], STAGE / name)
        staged[name] = hits[0]

train_csv = find_competition_file("train.csv")
if train_csv is None:
    raise SystemExit("train.csv not found -- attach the competition dataset")
shutil.copy(train_csv, STAGE / "train.csv")

# Rebind before anything imports a path out of config. The label modules read
# config.META_DIR at call time, so this is enough for the whole stack.
config.META_DIR = STAGE
config.TRAIN_CSV = STAGE / "train.csv"

print("staged into", STAGE)
for name in WANTED:
    where = staged.get(name)
    print(f"   {'[ok]  ' if where else '[none]'} {name:26s} {where or ''}")
print()
print("train.csv:", train_csv)

HAVE_DUMPS = all(n in staged for n in
                 ("llm_labels.parquet", "llm_labels_gold.parquet",
                  "llm_probs.parquet", "llm_probs_gold.parquet"))
HAVE_SHIPPED = "train_labels.parquet" in staged
print("extractor dumps attached:", HAVE_DUMPS)
if not HAVE_SHIPPED:
    raise SystemExit("train_labels.parquet not attached -- nothing to verify")

## A. Provenance — is this the file the 0.906 folds read?

An md5 and a fingerprint, because an upload date proves nothing: a re-upload
keeps the date and changes the bytes, and a stale copy keeps the bytes and
changes nothing visible.

The expected digest is the repo's `data/meta/train_labels.parquet` as of
2026-09-02. **A mismatch is not automatically a failure** — it means the
uploaded copy and the repo copy are different files, and which one is right
depends on which was uploaded last. It does mean stopping to find out before
trusting any number below.

The fingerprint underneath is the one recorded when `knee-labels` was first
checked: 4,349 rows, **zero gold studies** (the 58 must never appear in the
training matrix), and the distributional signature of the shipped labels.

In [ ]:
EXPECTED_MD5 = "164dba7c82c9519558f1e326e845f7eb"   # repo copy, 2026-09-02

shipped_path = STAGE / "train_labels.parquet"
digest = hashlib.md5(shipped_path.read_bytes()).hexdigest()
shipped = pd.read_parquet(shipped_path)
print("A. PROVENANCE")
print(f"   md5 {digest}")
print(f"       {'matches' if digest == EXPECTED_MD5 else 'DIFFERS FROM'} the repo copy"
      f" ({EXPECTED_MD5})")

gold = calibrate.load_gold(config.TRAIN_CSV)
gold_ids = set(gold[config.ID_COL])
leaked = shipped[config.ID_COL].isin(gold_ids).sum()
print()
print(f"   rows                 {len(shipped):,}      expected 4,349")
print(f"   gold studies present {leaked}          expected 0")
if leaked:
    print("   [FAIL] the gold 58 are in the training matrix -- every gold number")
    print("          in this project would be in-sample")
else:
    print("   [PASS] the gold 58 are held out of the training matrix")

## B. Reproduction — does the code still produce this file?

The step-1 check, applied to the label side. Nothing here decodes a pixel, so
unlike the cache verification this can afford **100% of the corpus** rather
than a 2% sample.

This is what "verified" means for a label pipeline, and it is worth being
precise about what it does *not* mean: a pipeline that is wrong in the same way
every time reproduces perfectly. Part D is where correctness is addressed, and
where it runs out of evidence.

In [ ]:
if not HAVE_DUMPS:
    print("B. REPRODUCTION")
    print("   [skip] extractor dumps not attached.")
    print("          Add llm_labels{,_gold}.parquet and llm_probs{,_gold}.parquet")
    print("          to knee-labels to enable this part.")
    rebuilt = None
else:
    rebuilt, fused_gold, extractor_aucs = fuse.build()
    from tools import audit_labels
    reproduced = audit_labels.check_reproduction(shipped, rebuilt,
                                                header="B. REPRODUCTION")

## C. Properties — do the documented numbers belong to this artifact?

Step 2 found the side classifier's documented 98.97% belonged to a run that no
longer existed anywhere. This is the same question asked of the labels, and it
reads the numbers off the file rather than out of the worklog.

In [ ]:
from tools import audit_labels

properties_ok = audit_labels.check_properties(
    shipped, header="C. DOCUMENTED PROPERTIES")

print()
print("   distinct target values per finding (ties are what AUC cannot order)")
for name in config.LABELS:
    print(f"      {name:18s} {shipped[name].nunique():5d}")

## D. Correctness — and how little the gold 58 can settle

The only external truth in this project is 58 radiologist-labelled studies,
1.3% of the corpus. Everything else — the fusion strategy, the lexicon blend,
the sharpening — is scored against them or against itself.

This part reports the macro AUC **with its bootstrap interval**, because the
point estimate on its own has been the source of more than one closed question
that the interval would have left open. Then it splits the gold by report
language, which is the specific worry step 3 raised, and finally states the
sample size that would be needed to resolve what the split cannot.

In [ ]:
print("D. CORRECTNESS, AND ITS RESOLUTION")
if not HAVE_DUMPS:
    print("   [skip] needs the extractor dumps to rebuild the gold-side targets")
else:
    L = config.LABELS
    truth = np.column_stack([gold[n].to_numpy().astype(int) for n in L])
    pred = np.column_stack([fused_gold[n].to_numpy(float) for n in L])

    def macro(t, p):
        scored = [calibrate._auc(t[:, i], p[:, i]) for i in range(t.shape[1])
                  if 0 < t[:, i].sum() < len(t)]
        return (float(np.mean(scored)), len(scored)) if scored else (np.nan, 0)

    def interval(t, p, draws=2000, seed=0):
        rng = np.random.default_rng(seed)
        values = []
        for _ in range(draws):
            idx = rng.integers(0, len(t), len(t))
            value, _ = macro(t[idx], p[idx])
            if not np.isnan(value):
                values.append(value)
        return np.array(values)

    point, scoreable = macro(truth, pred)
    draws = interval(truth, pred)
    lo, hi = np.percentile(draws, [2.5, 97.5])
    se = (hi - lo) / 3.92
    print(f"   gold studies {len(gold)}   slots {truth.size}"
          f"   positives {truth.sum()} ({100 * truth.mean():.1f}%)")
    print(f"   fused macro AUC {point:.3f}   95% CI [{lo:.3f}, {hi:.3f}]"
          f"   ({scoreable}/12 findings scoreable)")
    print(f"   implied macro SE {se:.3f}")

    # The language split. Detection is heuristic and refuses to guess -- see
    # tools/audit_labels.py -- so "unknown" is reported rather than folded
    # into either arm.
    language = gold["Report"].map(audit_labels.detect_language).to_numpy()
    english = language == "en"
    other = ~english & (language != "unknown")
    print()
    print(f"   English {english.sum()}   non-English {other.sum()}"
          f"   unknown {(language == 'unknown').sum()}")
    arms = {}
    for label, mask in (("English", english), ("non-English", other)):
        value, count = macro(truth[mask], pred[mask])
        arm = interval(truth[mask], pred[mask])
        arms[label] = arm
        alo, ahi = np.percentile(arm, [2.5, 97.5])
        print(f"      {label:12s} n={mask.sum():3d}  macro {value:.3f}"
              f"   95% CI [{alo:.3f}, {ahi:.3f}]")
    delta = arms["English"] - arms["non-English"]
    dlo, dhi = np.percentile(delta, [2.5, 97.5])
    print(f"      difference   {delta.mean():+.3f}   95% CI [{dlo:+.3f}, {dhi:+.3f}]")
    print(f"      P(English labels rank better) = {(delta > 0).mean():.2f}")

    # Threshold behaviour, which is where the lexicon's language-correlated
    # target shrinkage shows up even when the ranking does not move.
    print()
    for label, mask in (("English", english), ("non-English", other)):
        t, p = truth[mask], pred[mask]
        print(f"      {label:12s} slot agreement at 0.5 {100 * ((p > 0.5) == (t == 1)).mean():.1f}%"
              f"   positives recovered {100 * (p[t == 1] > 0.5).mean():.1f}%")

    # What this sample can and cannot see. The spread of the difference is
    # taken from its own bootstrap rather than scaled off the pooled SE --
    # the two arms are 28 and 23 studies, so the difference is noisier than
    # the whole, and using the pooled number here would flatter the test.
    # AUC's SE then falls as 1/sqrt(n), which fixes the rest by arithmetic.
    # Hanley-McNeil understates SE for correlated ratings, so every figure
    # below is optimistic: the real requirement is larger, not smaller.
    print()
    print("   RESOLUTION -- what would have to exist to answer this")
    gap_sd = float(delta.std())
    detectable = 2.8 * gap_sd
    print(f"      macro SE at n={len(gold)}: {se:.3f}")
    print(f"      SD of the English/non-English gap: {gap_sd:.3f}")
    print(f"      smallest gap detectable at 80% power: {detectable:.3f}")
    print(f"      observed gap: {abs(delta.mean()):.3f} -- inside the noise, so"
          " this split is underpowered rather than exonerating")
    for target in (0.010, 0.005):
        print(f"      gold studies for macro SE {target:.3f}:"
              f" {len(gold) * (se / target) ** 2:,.0f}")
    for target in (0.03, 0.02):
        needed = len(gold) * (gap_sd / (target / 2.8)) ** 2
        print(f"      gold studies to detect a {target:.2f} language gap:"
              f" {needed:,.0f}")
    print("      docs/plan.md P4 item 12 asks for >=300 studies; this is the"
          " arithmetic behind that number")

## E. Language reach — the mechanism behind the split above

The lexicon is one of the two signals whose *informativeness* sets every sample
weight, so its reach decides how much a study is allowed to teach, entirely
separately from what the study says. Reported length-matched, because
non-English reports are also shorter and the confound would otherwise carry the
whole result.

In [ ]:
if not HAVE_DUMPS:
    print("E. LANGUAGE REACH")
    print("   [skip] needs the extractor dumps")
else:
    reports = pd.read_csv(config.TRAIN_CSV)
    reports["language"] = reports["Report"].map(audit_labels.detect_language)
    unlabelled = calibrate.load_unlabelled(config.TRAIN_CSV)
    table = audit_labels.language_reach(reports, unlabelled,
                                        header="E. LANGUAGE REACH")
    audit_labels.stem_reach(reports)
    table.to_csv("/kaggle/working/label_language_reach.csv", index=False)
    print()
    print("wrote label_language_reach.csv")

## F. Against the gold images — are the labels actually right?

Part D asks how well the targets *order* studies, which is what the competition
metric rewards. It cannot say whether a label is right, and here the two come
apart.

These labels are derived from reports, and a report is not an inventory of what
the image contains. A finding the radiologist saw and did not write down is
simply absent from the text, and the pipeline can only write a zero for it. So
misses and false alarms are counted separately — a single error rate hides this
completely, because most slots are true negatives and calling everything
negative already looks good.

Broken out by the pipeline's own sample weight, which is the cut worth making:
the weight is assigned before anything sees the truth, so it either predicts
where the labels are wrong or it does not.

In [ ]:
if not HAVE_DUMPS:
    print("F. AGAINST THE GOLD IMAGES")
    print("   [skip] needs the extractor dumps to rebuild the gold-side targets")
else:
    per_finding = audit_labels.check_against_gold(gold, fused_gold)
    per_finding.to_csv("/kaggle/working/label_error_by_finding.csv", index=False)
    print()
    print("wrote label_error_by_finding.csv")

## Verdict

A and C are the checks that can fail a run. B is the reproduction guarantee. D
and E measure and do not judge — nothing in this notebook writes a label, and
changing the label matrix costs five fold retrains, which is a decision to
pre-register rather than a cell to run.

In [ ]:
print("VERDICT")
print(f"   A provenance   md5 {'matches repo' if digest == EXPECTED_MD5 else 'DIFFERS'},"
      f" gold leakage {leaked}")
if HAVE_DUMPS:
    print(f"   B reproduction {'PASS' if reproduced else 'FAIL'}")
else:
    print("   B reproduction skipped -- dumps not attached")
print(f"   C properties   {'PASS' if properties_ok else 'FAIL'}")
print("   D correctness  measured, not judged -- read the interval, not the point")
print("   F gold images  precision and recall, split apart -- see the table above")
print()
if digest != EXPECTED_MD5:
    print("Resolve the md5 difference before trusting anything downstream:")
    print("the uploaded matrix and the repo matrix are not the same file.")
else:
    print("The matrix that trained the 0.906 folds is the matrix this repo")
    print("reproduces. Correctness beyond that rests on 58 studies -- see D.")